# Experiment 5 — Is the physics-informed loss architecture-independent? Three generators that REPLACE the MONAI U-Net
**Why (RA's advice):** Experiments 1–4 tested the physics loss on ONE network (the published MONAI `DiffusionModelUNet`).
A reviewer will ask: *is the effect a property of the physics loss, or of that one network?*
Here the MONAI U-Net is **replaced** by three other generators, each trained **with and without** the physics loss:

| `MODEL` | generator | family | who runs it |
|---|---|---|---|
| `adm`  | ADM-style U-Net (Hugging Face `diffusers` `UNet2DModel`, AdaGN, class embedding) | diffusion, CNN | Account 1 |
| `dit`  | Diffusion Transformer DiT-B/8 (adaLN-Zero) | diffusion, transformer | Account 2 |
| `wgan` | the published repo's conditional WGAN-GP (`wcgan.py`, 256 px) | GAN (no diffusion) | Account 3 |

Everything else is **identical to Experiment 3**: 25 % of CBIS-DDSM train (same images), same augmentation, same physics module
(NLF σ²(μ) = b + aμ + cμ² fitted on real images, two-sided band [q0.1, q0.9]), same seeds 42–44, same evaluation
(200 images/class, oracle ConvNeXt, KID/FID, mammography-space KID / precision / recall, NPS, downstream ResNet-18).
Diffusion models use the same recipe loss and λ = 0.01; the GAN uses a gradient-balanced physics weight (ratio 0.1).

**Pre-registered (written before running):**
* PRIMARY: pooled over the 3 new generators × 3 seeds (9 pairs), relative change of KID in mammography feature space,
  one-sided Wilcoxon signed-rank (physics lower), α = 0.05.
* SECONDARY (Holm-corrected): precision in mammography space (higher), noise-power-spectrum distance (lower).
* Per-generator results (n = 3) and everything else are descriptive. Supplementary: + MONAI U-Net of Experiment 3 (n = 12).

**How to run (each account):** GPU **T4 ×2** · Internet **ON** · Add Input: *CBIS-DDSM Breast Cancer Image Dataset* and
*oracle-checkpoint-final* · Cell 1: set `MODEL` and `BUDGET_H` → **Save Version → Save & Run All (Commit)**.
Next session (if not finished): Add Input → *Your Work* → this notebook's latest version → Save & Run All again.
When Cell 6 prints `SHARE FILE COMPLETE`, download `share_<MODEL>.zip` from the Output tab and send it to the summary account.

In [ ]:
# ===================== CELL 1 — CONFIG (the ONLY cell you edit) =====================
# Experiment 5: the physics-informed loss on THREE OTHER GENERATORS that replace the MONAI U-Net.
# The SAME notebook is used by all three Kaggle accounts - only MODEL changes:
#   Account 1                     MODEL = "adm"      ADM-style U-Net (diffusers)      - diffusion
#   Account 2                     MODEL = "dit"      Diffusion Transformer DiT-B/8    - diffusion
#   Account 3                     MODEL = "wgan"     the repo's conditional WGAN-GP   - GAN
#   at the very end, one account  MODEL = "summary"  combine the three share zips (no GPU needed)
# Run it with "Save & Run All (Commit)" in every session. It continues automatically where it stopped.
MODEL = "adm"
BUDGET_H = 11.0          # hours THIS session may use = min(11.0, your remaining weekly GPU quota - 0.8)

# ---------------- fixed design - do NOT change (identical to Experiment 3) ----------------
SEEDS = [42, 43, 44]
DATA_FRAC = 0.25
LAM_DIFF = 0.01          # physics weight for the diffusion models (same as Exp 3)
LAM_GAN = 0.1            # GAN: physics gradient = 0.1 x adversarial gradient (gradient balancing)
MCFG = {
    "adm":  dict(kind="diffusion", lr=1e-5, bs=2, accum=4, epochs=45, log_every=100),   # same recipe as MONAI
    "dit":  dict(kind="diffusion", lr=1e-4, bs=2, accum=4, epochs=45, log_every=100),   # DiT paper learning rate
    "wgan": dict(kind="gan", lr=1e-4, bs=16, n_critic=5, gp=10.0, gan_iters=100, pair_h=2.3, log_every=50),
}
MAX_PAIR_H = 3.3         # safety cap: hours per seed pair for a diffusion model (MONAI took 2.4 h)
CLF_SEEDS = [0, 1, 2]    # downstream classifier repeats

import os, time, glob
assert MODEL in ("adm", "dit", "wgan", "summary"), 'MODEL must be "adm", "dit", "wgan" or "summary"'
BUDGET_H = min(float(BUDGET_H), 11.3)
if not os.path.exists("/tmp/pg_session_start"):
    open("/tmp/pg_session_start", "w").write(str(time.time()))
def used_h():
    return (time.time() - float(open("/tmp/pg_session_start").read())) / 3600
def find_any(name):
    fs = glob.glob(f"/kaggle/working/**/{name}", recursive=True) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return fs[0] if fs else None
def ARMS(s):
    lam = LAM_GAN if MODEL == "wgan" else LAM_DIFF
    return [(0, f"{MODEL}_base_s{s}", 0.0), (1, f"{MODEL}_pinn_s{s}", lam)]
print(f"MODEL = {MODEL} | budget {BUDGET_H} h | seeds {SEEDS} | data {DATA_FRAC}"
      + (f" | {MCFG[MODEL]}" if MODEL in MCFG else ""))

In [ ]:
# ===================== CELL 2 — setup (every session; needs Internet ON) =====================
import os, re, sys, subprocess, time
if MODEL == "summary":
    print("summary mode -> setup skipped (no GPU and no repository needed)")
else:
    os.chdir("/kaggle/working")
    if not os.path.isdir("/kaggle/working/U24_synthesis/.git"):
        subprocess.run("git clone -q https://github.com/Its-Me-Sk/U24_synthesis.git", shell=True, check=True)
    if not os.path.exists("/tmp/pip_done"):
        # Kaggle's image now runs Python 3.13: old pins (numpy 1.26.4 / pandas 2.2.2 force-reinstall) can no longer be
        # built there. Install the repo requirements once, then only fix what is really missing - never force-reinstall.
        subprocess.run("cd /kaggle/working/U24_synthesis && grep -v xformers requirements.txt > /tmp/req.txt && "
                       "pip install -q -r /tmp/req.txt", shell=True, check=True)
        subprocess.run("pip install -q --no-deps torch-fidelity", shell=True, check=True)
        open("/tmp/pip_done", "w").write("ok")
    # ---- verify the environment in a fresh process (this is what all training/evaluation scripts will see) ----
    probe = ("import sys, numpy, pandas, scipy, sklearn, torch, torchvision, torchmetrics, monai, generative, PIL; "
             "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
             "'| scipy', scipy.__version__, '| torch', torch.__version__, '| torchmetrics', torchmetrics.__version__, "
             "'| monai', monai.__version__)")
    env_chk = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if env_chk.returncode != 0:
        print(env_chk.stderr[-2500:])
        raise SystemExit("!! environment check failed (see the error above) -> send Claude this block")
    print("environment OK:", env_chk.stdout.strip())
    p = "/kaggle/working/U24_synthesis/lightning_synthesis/model_architectures/monai_ddpm.py"
    s = open(p).read()
    s2 = re.sub(r"use_flash_attention\s*=\s*True", "use_flash_attention=False", s)   # xformers not installed
    if s2 != s:
        open(p, "w").write(s2)
    m = "/kaggle/working/U24_synthesis/models.py"          # oracle loader: hard-coded config path of the original author
    s = open(m).read()
    s2 = re.sub(r"open\(\s*['\"]/home/[^'\"]*config\.yaml['\"]", "open('/kaggle/working/U24_synthesis/config.yaml'", s)
    if s2 != s:
        open(m, "w").write(s2)
    # Hugging Face diffusers (needed by the ADM model; installed on every account so the environments are identical)
    probe = [sys.executable, "-c", "import diffusers; from diffusers import UNet2DModel; print(diffusers.__version__)"]
    chk = subprocess.run(probe, capture_output=True, text=True)
    if chk.returncode != 0:
        try:
            import importlib.metadata as _md
            hub = _md.version("huggingface_hub")
        except Exception:
            hub = None
        # keep the installed huggingface_hub -> pip picks the newest diffusers that is compatible with it
        subprocess.run("pip install -q diffusers" + (f" huggingface_hub=={hub}" if hub else ""), shell=True)
        chk = subprocess.run(probe, capture_output=True, text=True)
    if chk.returncode == 0:
        print("diffusers", chk.stdout.strip(), "OK")
    else:
        print("!! diffusers could not be imported:", chk.stderr.strip()[-400:])
        if MODEL == "adm":
            raise SystemExit("MODEL = 'adm' needs diffusers -> check Internet is ON, then run again")
    print("Cell 2 done. Kernel restart NOT needed (all GPU work runs in subprocesses).")

In [ ]:
# ===================== CELL 3 — data cache + code + input checks (every session, no training) =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
os.makedirs(WD, exist_ok=True)
os.chdir(WD)

# ---------- the experiment code (written fresh every session) ----------
Path("pinn_physics.py").write_text(r'''# pinn_physics.py - physics-informed soft constraint (PINN-style residual) for mammogram DDPM.
# Physical model: signal-dependent noise-level function (NLF) of the X-ray film/digitiser chain
#   sigma^2(mu) = b + a*mu + c*mu^2
#   b: Gaussian read/digitiser noise, a*mu: Poisson (quantum) noise, c*mu^2: multiplicative film-grain/gain term
# The coefficients are CALIBRATED on real training images, then the residual between the NLF of a
# generated image and the physical NLF is added to the diffusion loss (lambda * residual).
import torch
import torch.nn.functional as F


class PhysicsNLF:
    def __init__(self, device, k=7, K=16, lo=0.08, hi=0.95, mask_lo=0.04):
        ax = torch.arange(7, device=device, dtype=torch.float32) - 3
        g = torch.exp(-ax ** 2 / 2.0)
        self.g = g / g.sum()
        self.k, self.K, self.lo, self.hi, self.mask_lo = k, K, lo, hi, mask_lo
        self.c = torch.linspace(lo, hi, K, device=device)
        self.bw = float(hi - lo) / (K - 1) * 0.6
        self.p = None                     # (b, a, c) of the calibrated NLF

    def blur(self, I):
        g = self.g
        I = F.conv2d(F.pad(I, (3, 3, 0, 0), mode="reflect"), g.view(1, 1, 1, 7))
        return F.conv2d(F.pad(I, (0, 0, 3, 3), mode="reflect"), g.view(1, 1, 7, 1))

    def stats(self, I):
        # I: (B,1,H,W) intensities in [0,1]. Returns per-image binned high-frequency variance vs local mean.
        h = I - self.blur(I)
        mu = F.avg_pool2d(I, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        v = F.avg_pool2d(h * h, self.k, 1, self.k // 2, count_include_pad=False).flatten(1)
        W = torch.exp(-(mu.unsqueeze(-1) - self.c) ** 2 / (2 * self.bw ** 2))
        W = W * (mu > self.mask_lo).unsqueeze(-1).float()        # ignore black background (air)
        mass = W.sum(1)                                           # (B,K)
        vbin = torch.einsum("bpk,bp->bk", W, v) / (mass + 1e-6)   # (B,K)
        return vbin, mass

    def model_var(self):
        b, a, c = self.p
        return (b + a * self.c + c * self.c ** 2).clamp_min(1e-7)

    def residual(self, I):
        # per-image physics residual: mass-weighted squared log-ratio of measured vs physical NLF
        vbin, mass = self.stats(I)
        r = (torch.log(vbin + 1e-5) - torch.log(self.model_var() + 1e-5)) ** 2
        w = mass / (mass.sum(1, keepdim=True) + 1e-6)
        return (w * r).sum(1)

    @torch.no_grad()
    def fit(self, X_u8, bs=32):
        dev = self.c.device
        tv = torch.zeros(self.K, device=dev, dtype=torch.float64)
        tm = torch.zeros(self.K, device=dev, dtype=torch.float64)
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            vb, m = self.stats(I)
            tv += (vb * m).sum(0).double()
            tm += m.sum(0).double()
        V = tv / tm.clamp_min(1e-6)
        wt = tm / tm.sum()
        c = self.c.double()

        def wls(cols):
            A = torch.stack(cols, 1)
            th = torch.linalg.solve(A.T @ (wt[:, None] * A), A.T @ (wt * V))
            pred = (A @ th).clamp_min(1e-9)
            ss_res = (wt * (V - pred) ** 2).sum()
            ss_tot = (wt * (V - (wt * V).sum()) ** 2).sum()
            lr = torch.log(V) - torch.log(pred)
            lres = (wt * lr ** 2).sum()
            ltot = (wt * (torch.log(V) - (wt * torch.log(V)).sum()) ** 2).sum()
            return th, float(1 - ss_res / ss_tot), float(1 - lres / ltot)

        th_lin, r2_lin, lr2_lin = wls([torch.ones_like(c), c])
        th_q, r2_q, lr2_q = wls([torch.ones_like(c), c, c ** 2])
        self.p = (float(th_q[0]), float(th_q[1]), float(th_q[2]))
        return {"model": "sigma^2 = b + a*mu + c*mu^2 (Poisson-Gaussian + multiplicative)",
                "b": self.p[0], "a": self.p[1], "c": self.p[2],
                "R2_var": r2_q, "R2_logvar": lr2_q,
                "linear_poisson_gaussian_only": {"b": float(th_lin[0]), "a": float(th_lin[1]),
                                                  "R2_var": r2_lin, "R2_logvar": lr2_lin},
                "bin_centers": [float(x) for x in c], "measured_var": [float(x) for x in V],
                "bin_weight": [float(x) for x in wt]}

    @torch.no_grad()
    def mean_residual(self, X_u8, bs=32):
        dev = self.c.device
        tot, n = 0.0, 0
        for i in range(0, X_u8.shape[0], bs):
            I = X_u8[i:i + bs].to(dev).float().div(255).unsqueeze(1)
            tot += float(self.residual(I).sum())
            n += I.shape[0]
        return tot / max(n, 1)

    def state(self):
        return {"p": list(self.p), "k": self.k, "K": self.K, "lo": self.lo, "hi": self.hi, "mask_lo": self.mask_lo}

    def load(self, s):
        self.p = tuple(s["p"])
''')
Path("gen_models.py").write_text(r'''# gen_models.py - Experiment 5: three generators that REPLACE the MONAI DiffusionModelUNet.
#   adm  : Hugging Face diffusers UNet2DModel (ADM / "improved DDPM" style CNN U-Net),
#          label enters through a CLASS EMBEDDING added to the timestep embedding (not cross-attention).
#   dit  : Diffusion Transformer (DiT-B/8, adaLN-Zero, Peebles & Xie 2023) - a transformer instead of a U-Net.
#   wgan : the published repo's own conditional WGAN-GP (wcgan.py), resized from 512 to 256 px.
# adm and dit are trained with EXACTLY the same diffusion recipe as the MONAI model (Experiments 3-4);
# only the denoising network changes. wgan is a different generative family (no diffusion).
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

NULL_LABEL = 2           # "no label" class used for classifier-free guidance (diffusion models only)
Z_DIM = 128              # WGAN latent size (repo default)
DIFFUSION_KINDS = ("adm", "dit")


# --------------------------------------------------------------------------------------------
# 1. ADM-style U-Net (diffusers)
# --------------------------------------------------------------------------------------------
def build_adm():
    from diffusers import UNet2DModel
    return UNet2DModel(
        sample_size=256, in_channels=1, out_channels=1, layers_per_block=2,
        block_out_channels=(64, 128, 256, 256, 512, 512),            # 256,128,64,32,16,8 px
        down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "DownBlock2D",
                          "AttnDownBlock2D", "AttnDownBlock2D"),      # self-attention at 16x16 and 8x8
        up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D",
                        "UpBlock2D", "UpBlock2D"),
        attention_head_dim=64, norm_num_groups=32, attn_norm_num_groups=32,
        resnet_time_scale_shift="scale_shift",                       # ADM's adaptive group norm (AdaGN)
        num_class_embeds=3)                                          # benign 0, malignant 1, null 2


# --------------------------------------------------------------------------------------------
# 2. Diffusion Transformer (DiT-B/8), written out in full (no external package)
# --------------------------------------------------------------------------------------------
def _sincos_1d(dim, pos):
    omega = 1.0 / 10000 ** (torch.arange(dim // 2, dtype=torch.float32) / (dim / 2.0))
    out = pos.flatten()[:, None] * omega[None]
    return torch.cat([out.sin(), out.cos()], dim=1)


def _sincos_2d(dim, n):
    g = torch.arange(n, dtype=torch.float32)
    gy, gx = torch.meshgrid(g, g, indexing="ij")
    return torch.cat([_sincos_1d(dim // 2, gy), _sincos_1d(dim // 2, gx)], dim=1)     # (n*n, dim)


def _t_embed(t, dim=256, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * freqs[None]
    return torch.cat([args.cos(), args.sin()], dim=1)


def _modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)


class DiTBlock(nn.Module):
    def __init__(self, h, heads):
        super().__init__()
        self.heads = heads
        self.n1 = nn.LayerNorm(h, elementwise_affine=False, eps=1e-6)
        self.qkv = nn.Linear(h, 3 * h)
        self.proj = nn.Linear(h, h)
        self.n2 = nn.LayerNorm(h, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(h, 4 * h), nn.GELU(approximate="tanh"), nn.Linear(4 * h, h))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(h, 6 * h))

    def forward(self, x, c):
        s1, sc1, g1, s2, sc2, g2 = self.ada(c).chunk(6, dim=1)
        B, N, H = x.shape
        q, k, v = self.qkv(_modulate(self.n1(x), s1, sc1)).view(B, N, 3, self.heads, H // self.heads).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, H)
        x = x + g1.unsqueeze(1) * self.proj(a)
        x = x + g2.unsqueeze(1) * self.mlp(_modulate(self.n2(x), s2, sc2))
        return x


class DiT(nn.Module):
    """DiT-B/8 on 1x256x256 pixels: 32x32 = 1024 tokens of 8x8 patches, hidden 768, 12 blocks, 12 heads."""
    def __init__(self, img=256, patch=8, in_ch=1, hidden=768, depth=12, heads=12, n_classes=3):
        super().__init__()
        self.p, self.c, self.g = patch, in_ch, img // patch
        self.x_emb = nn.Conv2d(in_ch, hidden, patch, patch)
        self.register_buffer("pos", _sincos_2d(hidden, self.g)[None], persistent=False)
        self.t_mlp = nn.Sequential(nn.Linear(256, hidden), nn.SiLU(), nn.Linear(hidden, hidden))
        self.y_emb = nn.Embedding(n_classes, hidden)
        self.blocks = nn.ModuleList([DiTBlock(hidden, heads) for _ in range(depth)])
        self.nf = nn.LayerNorm(hidden, elementwise_affine=False, eps=1e-6)
        self.ada_f = nn.Sequential(nn.SiLU(), nn.Linear(hidden, 2 * hidden))
        self.out = nn.Linear(hidden, patch * patch * in_ch)
        self._init()

    def _init(self):                                   # DiT paper initialisation (adaLN-Zero)
        def basic(m):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
        self.apply(basic)
        w = self.x_emb.weight.data
        nn.init.xavier_uniform_(w.view(w.shape[0], -1))
        nn.init.zeros_(self.x_emb.bias)
        nn.init.normal_(self.y_emb.weight, std=0.02)
        nn.init.normal_(self.t_mlp[0].weight, std=0.02)
        nn.init.normal_(self.t_mlp[2].weight, std=0.02)
        for b in self.blocks:
            nn.init.zeros_(b.ada[-1].weight); nn.init.zeros_(b.ada[-1].bias)
        nn.init.zeros_(self.ada_f[-1].weight); nn.init.zeros_(self.ada_f[-1].bias)
        nn.init.zeros_(self.out.weight); nn.init.zeros_(self.out.bias)

    def forward(self, x, t, y):
        B = x.shape[0]
        # residual stream kept in fp32 (under fp16 autocast only the matmuls run in half precision)
        h = self.x_emb(x).flatten(2).transpose(1, 2).float() + self.pos   # (B, 1024, hidden)
        c = self.t_mlp(_t_embed(t)).float() + self.y_emb(y).float()      # timestep + class conditioning
        for blk in self.blocks:
            h = blk(h, c)
        sh, sc = self.ada_f(c).chunk(2, dim=1)
        h = self.out(_modulate(self.nf(h), sh, sc))                # (B, 1024, p*p*C)
        h = h.view(B, self.g, self.g, self.p, self.p, self.c).permute(0, 5, 1, 3, 2, 4)
        return h.reshape(B, self.c, self.g * self.p, self.g * self.p)


# --------------------------------------------------------------------------------------------
# 3. Conditional WGAN-GP of the published repo, 256 px version
# --------------------------------------------------------------------------------------------
class Generator256(nn.Module):
    """Repo Generator (conditional-BN ResBlocks) with 6 instead of 7 up-sampling stages -> 1x256x256 in [-1,1]."""
    def __init__(self, z_dim=Z_DIM, emb_dim=128, n_classes=2):
        super().__init__()
        from model_architectures.wcgan import ResBlockG
        self.embed = nn.Embedding(n_classes, emb_dim)
        self.fc = nn.Linear(z_dim, 4 * 4 * 512)
        chs = [512, 512, 256, 128, 64, 32, 16]                     # 8,16,32,64,128,256 px
        self.blocks = nn.ModuleList([ResBlockG(chs[i], chs[i + 1], emb_dim) for i in range(6)])
        self.to_img = nn.Conv2d(16, 1, kernel_size=3, padding=1)

    def forward(self, z, y):
        y_emb = self.embed(y)
        h = self.fc(z).view(z.size(0), 512, 4, 4)
        for blk in self.blocks:
            h = F.interpolate(h, scale_factor=2, mode="nearest")
            h = blk(h, y_emb)
        return torch.tanh(self.to_img(F.relu(h)))


def build_wgan(base_ch=64):
    from model_architectures.wcgan import Discriminator
    G, D = Generator256(), Discriminator(n_classes=2, base_ch=base_ch)
    for m in list(G.modules()) + list(D.modules()):            # same initialisation as the repo
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            if hasattr(m, "weight") and m.weight is not None:
                nn.init.orthogonal_(m.weight, gain=0.8)
            if hasattr(m, "bias") and m.bias is not None:
                nn.init.constant_(m.bias, 0.0)
    return G, D


# --------------------------------------------------------------------------------------------
# common helpers
# --------------------------------------------------------------------------------------------
def build(kind):
    if kind == "adm":
        return build_adm()
    if kind == "dit":
        return DiT()
    if kind == "wgan":
        return build_wgan()
    raise ValueError(kind)


def eps_pred(net, kind, xt, t, y):
    """noise prediction of a diffusion denoiser; y = long labels (0, 1 or NULL_LABEL)."""
    if kind == "adm":
        return net(xt, t, class_labels=y).sample
    if kind == "dit":
        return net(xt, t, y)
    raise ValueError(kind)


def linear_alphas_cumprod(T=1000, b0=1e-4, b1=2e-2):
    """identical to MONAI DDPMScheduler(num_train_timesteps=1000) 'linear_beta' used by the published model."""
    betas = torch.linspace(b0, b1, T, dtype=torch.float32)
    return torch.cumprod(1.0 - betas, dim=0)


def n_params(m):
    return sum(p.numel() for p in m.parameters())
''')
Path("train_gen.py").write_text(r'''# train_gen.py - Experiment 5: train ONE arm (baseline or physics) of one REPLACEMENT generator.
#   --model adm | dit : conditional diffusion with the IDENTICAL recipe of Experiments 3-4 (25% data, loss
#                      0.7*MSE + 0.2*(1-MS-SSIM) + 0.05*HF on x0_hat, per-image min-max to [-1,1], shear/scale/flip,
#                      15% label dropout, batch 2 x grad-accum 4, Adam, clip 1.0, fp16). Only the denoiser changes.
#                      Physics arm: + lam * alpha_bar_t-weighted two-sided band hinge on x0_hat (lam = 0.01, as Exp 3).
#   --model wgan       : the repo's conditional WGAN-GP (5 critic steps per G step, gradient penalty 10,
#                      Adam betas (0.5, 0.999), critic lr = 0.5 x G lr, fp32).
#                      Physics arm: + lam_eff * band hinge on the GENERATED image. lam_eff is re-computed every step
#                      so that the physics gradient at the image is `lam` (= 0.1) times the adversarial gradient
#                      (gradient balancing, as the adaptive weight of Esser et al. 2021), because the critic's output
#                      scale is arbitrary and drifts - a fixed weight would be either negligible or dominant.
# The ONLY difference between the two arms of a model is the physics term (same seed -> same initial weights,
# same data order, same noise).
import os, sys, glob, time, json, argparse
import numpy as np
import torch
import torch.nn.functional as F

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
from pinn_physics import PhysicsNLF
import gen_models as GM

ap = argparse.ArgumentParser()
ap.add_argument("--model", required=True, choices=["adm", "dit", "wgan"])
ap.add_argument("--arm", required=True)                  # e.g. adm_base_s42 | adm_pinn_s42
ap.add_argument("--lam", type=float, default=0.0)        # 0 = baseline
ap.add_argument("--data_frac", type=float, default=0.25)
ap.add_argument("--band_lo", type=float, default=0.1)
ap.add_argument("--band_hi", type=float, default=0.9)
ap.add_argument("--epochs", type=int, default=45)
ap.add_argument("--bs", type=int, default=2)
ap.add_argument("--accum", type=int, default=4)          # diffusion only
ap.add_argument("--lr", type=float, default=1e-5)
ap.add_argument("--p_uncond", type=float, default=0.15)  # diffusion only (classifier-free guidance)
ap.add_argument("--n_critic", type=int, default=5)       # GAN only
ap.add_argument("--gp", type=float, default=10.0)        # GAN only
ap.add_argument("--gan_iters", type=int, default=100)    # GAN only: generator steps per "epoch"
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--budget_h", type=float, default=11.0)
ap.add_argument("--smoke_batches", type=int, default=0)
ap.add_argument("--log_every", type=int, default=100)
ap.add_argument("--ckpt_every", type=int, default=5)     # full (resumable) checkpoint every N epochs
ap.add_argument("--final", action="store_true")          # on completion: write <arm>_done.json, delete resume file
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
a = ap.parse_args()
GAN = a.model == "wgan"

TAG = f"[{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)

try:
    T0 = float(open("/tmp/pg_session_start").read())
except Exception:
    T0 = time.time()
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
os.makedirs(a.ckpt_dir, exist_ok=True)

def save_atomic(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "rb") as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, path)

def minmax_pm1(x):                                   # per-image min-max -> [-1,1] (same as Exp 3)
    mn = x.amin(dim=(1, 2, 3), keepdim=True)
    mx = x.amax(dim=(1, 2, 3), keepdim=True)
    return 2 * (x - mn) / (mx - mn + 1e-8) - 1

_sob = torch.tensor([[1, 0, -1], [2, 0, -2], [1, 0, -1]], dtype=torch.float32)
KX = (_sob[None, None] / 8).to(dev)
KY = (_sob.t()[None, None] / 8).to(dev)
def hf_loss(x, y):                                   # original HighFreqLoss of monai_ddpm.py
    gx = F.conv2d(x, KX, padding=1) - F.conv2d(y, KX, padding=1)
    gy = F.conv2d(x, KY, padding=1) - F.conv2d(y, KY, padding=1)
    return (gx.pow(2) + gy.pow(2)).mean()

# ---------------- data: the SAME 25% stratified subset as Experiments 3-4 (rng 123) ----------------
d = np.load("/tmp/train_cache.npz")
X = torch.from_numpy(d["x"])
Y = torch.from_numpy(d["y"]).long()
_rng = np.random.default_rng(123)
_keep = []
for _c in (0, 1):
    _idx = np.where(d["y"] == _c)[0]
    _keep += sorted(_rng.choice(_idx, int(round(a.data_frac * len(_idx))), replace=False).tolist())
_keep = np.array(sorted(_keep))
X, Y = X[_keep], Y[_keep]
N = X.shape[0]
Xg, Yg = X.to(dev), Y.to(dev)
log(f"data (frac={a.data_frac}) {N} images {tuple(X.shape[1:])} benign={int((Y == 0).sum())} "
    f"malignant={int((Y == 1).sum())} on {torch.cuda.get_device_name(0)}")

def augment(img, gg):
    # img: (B,1,H,W) float in 0..255. shear +-0.1, scale +-5% (p=0.9), h-flip p=0.5 (same as Exp 3)
    B = img.shape[0]
    do = (torch.rand(B, generator=gg, device=dev) < 0.9).float()
    u = lambda: (torch.rand(B, generator=gg, device=dev) * 2 - 1)
    sx, sy, shx, shy = 1 + 0.05 * u() * do, 1 + 0.05 * u() * do, 0.1 * u() * do, 0.1 * u() * do
    zero = torch.zeros(B, device=dev)
    theta = torch.stack([torch.stack([sx, shx, zero], 1), torch.stack([shy, sy, zero], 1)], 1)
    grid = F.affine_grid(theta, list(img.shape), align_corners=False)
    out = F.grid_sample(img, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    flip = torch.rand(B, generator=gg, device=dev) < 0.5
    return torch.where(flip.view(-1, 1, 1, 1), out.flip(-1), out)

def real_batch(idx, gg):
    return minmax_pm1(augment(Xg[idx].float().unsqueeze(1), gg)), Yg[idx]

# ---------------- model ----------------
torch.manual_seed(a.seed)                            # identical initial weights for both arms
if GAN:
    G, D = GM.build_wgan()
    G, D = G.to(dev), D.to(dev)
    nets = {"G": G, "D": D}
    opts = {"G": torch.optim.Adam(G.parameters(), lr=a.lr, betas=(0.5, 0.999)),
            "D": torch.optim.Adam(D.parameters(), lr=0.5 * a.lr, betas=(0.5, 0.999))}
    scaler = None
    main_net = G
else:
    net = GM.build(a.model).to(dev)
    nets = {"net": net}
    opts = {"net": torch.optim.Adam(net.parameters(), lr=a.lr)}
    scaler = torch.amp.GradScaler("cuda")
    ac = GM.linear_alphas_cumprod().to(dev)
    T = ac.numel()
    main_net = net
log(f"model {a.model}: " + ", ".join(f"{k} {GM.n_params(m) / 1e6:.1f}M params" for k, m in nets.items()))
nlf = PhysicsNLF(dev)
tau_lo = tau_hi = None

def full_state(epoch, step, opt_steps, history):
    return {"models": {k: m.state_dict() for k, m in nets.items()},
            "opts": {k: o.state_dict() for k, o in opts.items()},
            "scaler": scaler.state_dict() if scaler is not None else None,
            "epoch": epoch, "step": step, "opt_steps": opt_steps, "args": vars(a), "nlf": nlf.state(),
            "tau_lo": tau_lo, "tau_hi": tau_hi, "history": history}

# ---------------- resume (newest of /kaggle/working and attached earlier outputs) ----------------
paths = [os.path.join(a.ckpt_dir, f"{a.arm}_last.pt")]
if not a.smoke_batches:
    paths += glob.glob(f"/kaggle/input/**/{a.arm}_last.pt", recursive=True)
cands = []
for p in paths:
    if os.path.exists(p):
        try:
            ck = torch.load(p, map_location="cpu", weights_only=False, mmap=True)
            cands.append((int(ck["step"]), p))
            del ck
        except Exception as e:
            log("unreadable checkpoint (skipped):", p, repr(e)[:150])
start_epoch, step, opt_steps, history, resumed = 0, 0, 0, [], None
if cands:
    resumed = max(cands)[1]
    ck = torch.load(resumed, map_location="cpu", weights_only=False)
    for k in ("model", "lam", "lr", "bs", "accum", "seed", "p_uncond", "data_frac", "band_lo", "band_hi",
              "n_critic", "gp", "gan_iters"):
        if ck["args"].get(k) != getattr(a, k):
            log(f"!! WARNING {k} differs: checkpoint={ck['args'].get(k)} now={getattr(a, k)}")
    for k, m in nets.items():
        m.load_state_dict(ck["models"][k])
    for k, o in opts.items():
        o.load_state_dict(ck["opts"][k])
    if scaler is not None and ck.get("scaler"):
        scaler.load_state_dict(ck["scaler"])
    start_epoch, step, opt_steps, history = int(ck["epoch"]), int(ck["step"]), int(ck["opt_steps"]), ck["history"]
    nlf.load(ck["nlf"])
    tau_lo, tau_hi = float(ck["tau_lo"]), float(ck["tau_hi"])
    del ck
    torch.cuda.empty_cache()

# ---------------- physics calibration on REAL images (identical for every model and arm) ----------------
Xf = X.float()
mn = Xf.amin(dim=(1, 2), keepdim=True); mx = Xf.amax(dim=(1, 2), keepdim=True)
Xmm = ((Xf - mn) / (mx - mn + 1e-8) * 255).round().clamp(0, 255).to(torch.uint8)
del Xf
rep = None
if nlf.p is None:
    rep = nlf.fit(Xmm)
    log(f"NLF fit: sigma^2 = {rep['b']:.3e} + {rep['a']:.3e}*mu + {rep['c']:.3e}*mu^2 | "
        f"R2(var)={rep['R2_var']:.3f} R2(logvar)={rep['R2_logvar']:.3f}")
with torch.no_grad():
    res_real = torch.cat([nlf.residual(Xmm[i:i + 32].to(dev).float().div(255).unsqueeze(1)).cpu()
                          for i in range(0, N, 32)])
if tau_lo is None:
    tau_lo = float(torch.quantile(res_real, a.band_lo))
    tau_hi = float(torch.quantile(res_real, a.band_hi))
    with open(os.path.join(a.ckpt_dir, f"nlf_fit_{a.arm}.json"), "w") as fh:
        json.dump({**rep, "tau_lo": tau_lo, "tau_hi": tau_hi, "band": [a.band_lo, a.band_hi],
                   "real_mean_residual": float(res_real.mean())}, fh, indent=2)
log(f"physics residual of REAL images: mean={float(res_real.mean()):.4f} | "
    f"band [q{a.band_lo}, q{a.band_hi}] = [{tau_lo:.4f}, {tau_hi:.4f}]")

def band_hinge(r):
    return F.relu(r - tau_hi) + F.relu(tau_lo - r)

n_opt = sum(1 for o in opts.values() for s in o.state.values() if s)
chk = float(sum(p.detach().float().sum() for p in list(main_net.parameters())[:5]))
log(f"START epoch={start_epoch} step={step} opt_states={n_opt} lam={a.lam} checksum={chk:.6f} "
    f"resumed={resumed or 'scratch'}")
if resumed and n_opt == 0:
    log("!! WARNING: checkpoint loaded but optimizer state is EMPTY -> not a true resume")

spe_full = a.gan_iters if GAN else N // a.bs
spe = min(spe_full, a.smoke_batches) if a.smoke_batches else spe_full
durs = [h["minutes"] * 60 for h in history[-3:]]
reason = "max epochs reached"
bad = 0
ZERO = torch.zeros((), device=dev)
timing = None

def diffusion_step(i, perm, gg):
    idx = perm[i * a.bs:(i + 1) * a.bs].to(dev)
    x0, y = real_batch(idx, gg)
    t = torch.randint(0, T, (a.bs,), generator=gg, device=dev)
    eps = torch.randn(x0.shape, generator=gg, device=dev)
    drop = torch.rand(a.bs, generator=gg, device=dev) < a.p_uncond
    ylab = torch.where(drop, torch.full_like(y, GM.NULL_LABEL), y)
    ab = ac[t].view(-1, 1, 1, 1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    with torch.autocast("cuda", dtype=torch.float16):
        eps_hat = GM.eps_pred(net, a.model, xt, t, ylab)
    eps_hat = eps_hat.float()
    mse = F.mse_loss(eps_hat, eps)
    x0_hat = ((xt - (1 - ab).sqrt() * eps_hat) / ab.sqrt()).clamp(-1, 1)
    ms = 1.0 - msssim((x0_hat + 1) / 2, (x0 + 1) / 2, data_range=2.0, gaussian_kernel=False, normalize="relu")
    hf = hf_loss(x0_hat, x0)
    comp = 0.7 * mse + 0.2 * ms + 0.05 * hf                       # exact original loss
    w = ab.view(-1)
    if a.lam > 0:
        r = nlf.residual((x0_hat + 1) / 2)
        hinge = (w * band_hinge(r)).mean()
        loss = comp + a.lam * hinge
    else:
        with torch.no_grad():                                     # baseline: tracked, never trained on
            r = nlf.residual((x0_hat + 1) / 2)
            hinge = (w * band_hinge(r)).mean()
        loss = comp
    phys = (w * r).mean().detach()
    return loss, {"main": comp, "mse": mse, "ms": ms, "hf": hf, "phys": phys, "hinge": hinge}

def gan_step(gg):
    # ---- critic: n_critic updates with gradient penalty (Gulrajani et al. 2017) ----
    for _ in range(a.n_critic):
        idx = torch.randint(0, N, (a.bs,), generator=gg, device=dev)
        x_real, y = real_batch(idx, gg)
        z = torch.randn(a.bs, GM.Z_DIM, generator=gg, device=dev)
        with torch.no_grad():
            x_fake = G(z, y)
        d_real = D(x_real, y).mean()
        d_fake = D(x_fake, y).mean()
        e = torch.rand(a.bs, 1, 1, 1, generator=gg, device=dev)
        xi = (e * x_real + (1 - e) * x_fake).requires_grad_(True)
        gr = torch.autograd.grad(D(xi, y).sum(), xi, create_graph=True)[0]
        gp = (gr.flatten(1).norm(2, dim=1) - 1).pow(2).mean()
        d_loss = d_fake - d_real + a.gp * gp
        if not torch.isfinite(d_loss):
            return None, None
        opts["D"].zero_grad(set_to_none=True)
        d_loss.backward()
        opts["D"].step()
    # ---- generator: 1 update (critic frozen) ----
    for p in D.parameters():
        p.requires_grad_(False)
    idx = torch.randint(0, N, (a.bs,), generator=gg, device=dev)
    y = Yg[idx]                                                  # labels follow the data distribution
    z = torch.randn(a.bs, GM.Z_DIM, generator=gg, device=dev)
    x_fake = G(z, y)
    adv = -D(x_fake, y).mean()
    if a.lam > 0:
        r = nlf.residual((x_fake + 1) / 2)
        hinge = band_hinge(r).mean()
        g_adv = torch.autograd.grad(adv, x_fake, retain_graph=True)[0]
        g_phy = torch.autograd.grad(hinge, x_fake, retain_graph=True)[0]
        n_phy = float(g_phy.norm())
        lam_eff = (a.lam * float(g_adv.norm()) / n_phy) if n_phy > 1e-12 else 0.0
        lam_eff = min(lam_eff, 1e4)
        loss = adv + lam_eff * hinge
    else:
        with torch.no_grad():                                     # baseline: tracked, never trained on
            r = nlf.residual((x_fake.detach() + 1) / 2)
            hinge = band_hinge(r).mean()
        lam_eff = 0.0
        loss = adv
    ok = bool(torch.isfinite(loss))
    if ok:
        opts["G"].zero_grad(set_to_none=True)
        loss.backward()
        opts["G"].step()
    for p in D.parameters():
        p.requires_grad_(True)
    if not ok:
        return None, None
    return loss, {"main": adv, "wdist": (d_real - d_fake).detach(), "gp": gp.detach(), "phys": r.mean().detach(),
                  "hinge": hinge.detach(), "lam_eff": torch.tensor(lam_eff)}

if not GAN:
    from torchmetrics.functional.image import multiscale_structural_similarity_index_measure as msssim

KEYS = ["loss", "main", "mse", "ms", "hf", "wdist", "gp", "lam_eff", "phys", "hinge"]
SHOW = ["loss", "main", "phys", "hinge"] + (["wdist", "gp", "lam_eff"] if GAN else ["mse", "ms", "hf"])
for epoch in range(start_epoch, a.epochs):
    used_h = (time.time() - T0) / 3600
    if durs and used_h + 1.15 * max(durs[-3:]) / 3600 + 0.2 > a.budget_h:
        reason = f"time budget ({used_h:.2f} h used of {a.budget_h} h)"
        if epoch > start_epoch and epoch % a.ckpt_every != 0:   # make sure the newest epoch is resumable
            save_atomic(full_state(epoch, step, opt_steps, history), os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
            log(f"budget stop: saved resumable checkpoint at epoch {epoch}")
        break
    gcpu = torch.Generator().manual_seed(a.seed * 1000 + epoch)
    perm = torch.randperm(N, generator=gcpu)
    gg = torch.Generator(device=dev).manual_seed(a.seed * 1000 + epoch)
    for m in nets.values():
        m.train()
    if not GAN:
        opts["net"].zero_grad(set_to_none=True)
    t_ep, t_win = time.time(), time.time()
    S = {k: 0.0 for k in KEYS}
    n_ok = 0
    t_mark = None
    for i in range(spe):
        if i == 10 and a.smoke_batches:
            torch.cuda.synchronize(); t_mark = time.time()
        if GAN:
            loss, parts = gan_step(gg)
            if loss is None:
                bad += 1
                if bad > 50:
                    raise RuntimeError("50 consecutive non-finite losses - stopping")
                continue
            step += 1
            opt_steps += 1
        else:
            loss, parts = diffusion_step(i, perm, gg)
            if not torch.isfinite(loss):
                bad += 1
                if bad > 50:
                    raise RuntimeError("50 consecutive non-finite losses - stopping")
                continue
            scaler.scale(loss / a.accum).backward()
            step += 1
            if (i + 1) % a.accum == 0 or i == spe - 1:
                scaler.unscale_(opts["net"])
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(opts["net"])
                scaler.update()
                opts["net"].zero_grad(set_to_none=True)
                opt_steps += 1
        bad = 0
        n_ok += 1
        S["loss"] += float(loss.detach())
        for k, v in parts.items():
            S[k] += float(v)
        if (i + 1) % a.log_every == 0:
            ips = a.log_every / (time.time() - t_win)
            t_win = time.time()
            m = {k: v / n_ok for k, v in S.items()}
            log(f"ep {epoch + 1} it {i + 1}/{spe} step {step} " + " ".join(f"{k} {m[k]:.4f}" for k in SHOW)
                + f" | {ips:.2f} it/s")
    if t_mark is not None and spe > 12:
        torch.cuda.synchronize()
        timing = {"sec_per_it": (time.time() - t_mark) / (spe - 10), "spe_full": spe_full, "bs": a.bs,
                  "peak_mem_gb": torch.cuda.max_memory_allocated() / 1e9, "model": a.model, "arm": a.arm}
        with open(os.path.join(a.ckpt_dir, f"timing_{a.arm}.json"), "w") as fh:
            json.dump(timing, fh, indent=2)
        log(f"TIMING sec_per_it={timing['sec_per_it']:.4f} (full epoch = {spe_full} it -> "
            f"{timing['sec_per_it'] * spe_full / 60:.1f} min)")
    dur = time.time() - t_ep
    durs.append(dur)
    m = {k: v / max(n_ok, 1) for k, v in S.items()}
    history.append({"epoch": epoch + 1, "step": step, "opt_steps": opt_steps, **m, "minutes": dur / 60})
    if (epoch + 1) % a.ckpt_every == 0 or epoch + 1 == a.epochs:
        save_atomic(full_state(epoch + 1, step, opt_steps, history), os.path.join(a.ckpt_dir, f"{a.arm}_last.pt"))
    with open(os.path.join(a.ckpt_dir, f"{a.arm}_history.json"), "w") as fh:
        json.dump(history, fh, indent=1)
    left = a.epochs - epoch - 1
    log(f"SAVED epoch={epoch + 1} step={step} opt_steps={opt_steps} " + " ".join(f"{k}={m[k]:.4f}" for k in SHOW)
        + f" | {dur / 60:.1f} min/epoch | peak GPU mem {torch.cuda.max_memory_allocated() / 1e9:.1f} GB | "
          f"~{left * dur / 3600:.1f} h for {left} more")

done_ep = history[-1]["epoch"] if history else start_epoch
if done_ep >= a.epochs:
    gen_key = "G" if GAN else "net"
    sd = nets[gen_key].state_dict()
    if not GAN:                                                  # diffusion weights stored in fp16 (half the size)
        sd = {k: (v.detach().half() if v.is_floating_point() else v) for k, v in sd.items()}
    save_atomic({"models": {gen_key: {k: v.detach().cpu() for k, v in sd.items()}}, "model": a.model,
                 "epoch": done_ep, "step": step, "arm": a.arm, "lam": a.lam, "nlf": nlf.state(),
                 "tau_lo": tau_lo, "tau_hi": tau_hi}, os.path.join(a.ckpt_dir, f"{a.arm}_w_final.pt"))
    log(f"final weights -> {a.arm}_w_final.pt")
    if a.final:
        with open(os.path.join(a.ckpt_dir, f"{a.arm}_done.json"), "w") as fh:
            json.dump({"arm": a.arm, "model": a.model, "epochs": done_ep, "step": step, "opt_steps": opt_steps,
                       "args": vars(a)}, fh, indent=2)
        p = os.path.join(a.ckpt_dir, f"{a.arm}_last.pt")
        if os.path.exists(p):
            os.remove(p)
        log("final: wrote done marker, removed resume file (final weights kept)")
log(f"FINISHED epochs_completed={done_ep} step={step} reason={reason}")
''')
Path("eval_gen.py").write_text(r'''# eval_gen.py - Experiment 5: sample one trained arm and score it with EXACTLY the protocol of Experiments 3-4.
#   diffusion (adm, dit): DDIM 50 steps + classifier-free guidance 3.0 (null label = 2)
#   wgan                : G(z, label), one forward pass
# 200 images per class, same random seeds for both arms of a model (sample i of baseline and of physics start
# from the same noise -> paired comparison). Metrics: oracle class accuracy, KID / precision / recall in the
# oracle's mammography feature space, Inception KID / FID / precision / recall, NLF physics residual gap,
# noise-power-spectrum (NPS) distance (independent physics metric, never used in training).
import os, sys, glob, json, time, argparse
import numpy as np
import torch

WD = "/kaggle/working/U24_synthesis/lightning_synthesis"
sys.path.insert(0, WD)
from pinn_physics import PhysicsNLF
import gen_models as GM

ap = argparse.ArgumentParser()
ap.add_argument("--arm", required=True)                  # e.g. adm_pinn_s42
ap.add_argument("--n_per_class", type=int, default=200)
ap.add_argument("--steps", type=int, default=50)
ap.add_argument("--gs", type=float, default=3.0)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--seed", type=int, default=2025)
ap.add_argument("--require_oracle", action="store_true")
ap.add_argument("--ckpt_dir", default="/kaggle/working/ckpt")
ap.add_argument("--out", default="/kaggle/working/eval")
a = ap.parse_args()
TAG = f"[eval-{a.arm}]"
def log(*m):
    print(TAG, *m, flush=True)
dev = torch.device("cuda")
os.makedirs(a.out, exist_ok=True)

name = f"{a.arm}_w_final.pt"
found = glob.glob(os.path.join(a.ckpt_dir, name)) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
if not found:
    raise SystemExit(f"{TAG} {name} not found in {a.ckpt_dir} or /kaggle/input")
snap = torch.load(found[0], map_location="cpu", weights_only=False)
kind = snap["model"]
log(f"weights {found[0]} | model={kind} epoch={snap['epoch']} step={snap['step']} lam={snap['lam']}")
torch.manual_seed(0)
if kind == "wgan":
    net, _D = GM.build_wgan()
    del _D
    net.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in snap["models"]["G"].items()})
else:
    net = GM.build(kind)
    net.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in snap["models"]["net"].items()})
net = net.to(dev).eval()
n_par = GM.n_params(net)
nlf = PhysicsNLF(dev)
nlf.load(snap["nlf"])
ac = GM.linear_alphas_cumprod().to(dev)
T = ac.numel()

@torch.no_grad()
def ddim(label, n, gen):
    x = torch.randn(n, 1, 256, 256, generator=gen, device=dev)
    ts = torch.linspace(T - 1, 0, a.steps, device=dev).round().long()
    y2 = torch.cat([torch.full((n,), GM.NULL_LABEL, device=dev, dtype=torch.long),
                    torch.full((n,), label, device=dev, dtype=torch.long)])
    for i, t in enumerate(ts):
        ab_t = ac[t]
        ab_prev = ac[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0, device=dev)
        tb = torch.full((2 * n,), int(t), device=dev, dtype=torch.long)
        with torch.autocast("cuda", dtype=torch.float16):
            out = GM.eps_pred(net, kind, torch.cat([x, x]), tb, y2)
        eu, ec = out.float().chunk(2)
        eps = eu + a.gs * (ec - eu)
        x0 = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1, 1)
        eps = (x - ab_t.sqrt() * x0) / (1 - ab_t).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev).sqrt() * eps
    return ((x + 1) / 2).clamp(0, 1)

@torch.no_grad()
def gan(label, n, gen):
    z = torch.randn(n, GM.Z_DIM, generator=gen, device=dev)
    y = torch.full((n,), label, device=dev, dtype=torch.long)
    return ((net(z, y).float() + 1) / 2).clamp(0, 1)

sampler = gan if kind == "wgan" else ddim

# ---------------- generate ----------------
t0 = time.time()
imgs, labels = [], []
for label in (0, 1):
    done, b = 0, 0
    while done < a.n_per_class:
        n = min(a.bs, a.n_per_class - done)
        gen = torch.Generator(device=dev).manual_seed(a.seed + 100000 * label + b)
        imgs.append(sampler(label, n, gen).cpu())
        labels += [label] * n
        done += n
        b += 1
    log(f"class {label}: {done} samples ({(time.time() - t0) / 60:.1f} min)")
G = torch.cat(imgs)                                    # (M,1,256,256) in [0,1]
L = torch.tensor(labels)
t_sample = (time.time() - t0) / 60

# ---------------- real test images ----------------
d = np.load("/tmp/test_cache.npz")
R = torch.from_numpy(d["x"]).float().div(255).unsqueeze(1)
RL = torch.from_numpy(d["y"]).long()

# ---------------- oracle classifier (ConvNeXt-Tiny trained on real CBIS-DDSM) ----------------
root = "/kaggle/working/U24_synthesis"
sys.path.insert(0, root)
cwd = os.getcwd(); os.chdir(root)
from models import get_model
os.chdir(cwd)
op = glob.glob("/kaggle/input/**/oracle-checkpoint-final/last.pt", recursive=True)
if not op and a.require_oracle:
    raise SystemExit(f"{TAG} oracle-checkpoint-final/last.pt not attached -> Add Input -> oracle-checkpoint-final")
res = {"arm": a.arm, "model": kind, "n_params": n_par, "epoch": snap["epoch"], "step": snap["step"],
       "lam": snap["lam"], "n_per_class": a.n_per_class, "ddim_steps": a.steps if kind != "wgan" else None,
       "guidance": a.gs if kind != "wgan" else None, "sampling_minutes": t_sample}
correct = np.zeros(len(L), dtype=bool)
if op:
    orc = get_model("convnext_tiny", pretrained=False)
    orc.load_state_dict(torch.load(op[0], map_location="cpu", weights_only=False)["model_state"])
    orc = orc.half().to(dev).eval()
    last_lin = [m for m in orc.modules() if isinstance(m, torch.nn.Linear)][-1]
    _feat = {}
    last_lin.register_forward_hook(lambda mod, inp, out: _feat.__setitem__("f", inp[0].detach().float().flatten(1).cpu()))
    FEAT = {}
    def oracle_pred(I01, key):
        out, feats = [], []
        for i in range(0, len(I01), 32):
            x = I01[i:i + 32].to(dev)
            mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
            x = ((x - mn) / (mx - mn + 1e-8) - 0.5) / 0.5
            with torch.no_grad():
                out.append(orc(x.half()).argmax(1).cpu())
            feats.append(_feat["f"])
        FEAT[key] = torch.cat(feats)
        return torch.cat(out)
    pr = oracle_pred(R, "real")
    res["oracle_real_acc"] = {c: float((pr[RL == k] == k).float().mean()) for k, c in enumerate(["benign", "malignant"])}
    pg = oracle_pred(G, "gen")
    correct = (pg == L).numpy()
    res["oracle_acc"] = {c: float(correct[L.numpy() == k].mean()) for k, c in enumerate(["benign", "malignant"])}
    res["oracle_acc"]["mean"] = float(correct.mean())
    log(f"oracle on REAL test: {res['oracle_real_acc']} | on generated: {res['oracle_acc']}")
else:
    log("oracle checkpoint not found (attach oracle-checkpoint-final) - skipping oracle metrics")

# ---------------- physics residual (generated vs real) ----------------
def all_res(I):
    return torch.cat([nlf.residual(I[i:i + 32].to(dev)).cpu() for i in range(0, len(I), 32)])
rG, rR = all_res(G), all_res(R)
res["physics_residual_generated"] = float(rG.mean())
res["physics_residual_real_test"] = float(rR.mean())
res["physics_abs_gap_to_real"] = abs(float(rG.mean()) - float(rR.mean()))
res["physics_residual_generated_std"] = float(rG.std())
res["physics_residual_real_std"] = float(rR.std())
res["frac_inside_band"] = float(((rG >= snap["tau_lo"]) & (rG <= snap["tau_hi"])).float().mean())
log(f"physics residual: generated={res['physics_residual_generated']:.4f} real={res['physics_residual_real_test']:.4f} "
    f"| inside real band: {100 * res['frac_inside_band']:.1f}%")

# ---------------- noise power spectrum (independent physics metric) ----------------
def raps(xu8):
    x = xu8.astype(np.float32)
    mn = x.min(axis=(1, 2), keepdims=True); mx = x.max(axis=(1, 2), keepdims=True)
    x = (x - mn) / (mx - mn + 1e-8); x = x - x.mean(axis=(1, 2), keepdims=True)
    P = np.abs(np.fft.fftshift(np.fft.fft2(x), axes=(1, 2))) ** 2
    h, w = x.shape[1:]; yy, xx = np.indices((h, w)); r = np.hypot(yy - h / 2, xx - w / 2).astype(int)
    nb = min(h, w) // 2
    out = np.stack([np.bincount(r.ravel(), weights=p.ravel(), minlength=r.max() + 1)[:nb] /
                    np.bincount(r.ravel(), minlength=r.max() + 1)[:nb] for p in P])
    return np.log10(out.mean(0) + 1e-12)
Gu8 = (G[:, 0].numpy() * 255).astype(np.uint8)           # same conversion as Experiments 3-4
r_gen, r_real = raps(Gu8), raps(d["x"])
res["NPS_dist"] = float(np.mean((r_gen[32:128] - r_real[32:128]) ** 2))
res["raps_gen"] = [float(v) for v in r_gen]
res["raps_real"] = [float(v) for v in r_real]
log(f"noise-power-spectrum distance to real = {res['NPS_dist']:.4f}")

# ---------------- feature-space metrics (KID polynomial kernel, k-NN precision/recall) ----------------
def kid_feats(fr, ff, n_sub=100, sub=100, seed=0):
    fr, ff = fr.double(), ff.double()
    dd = fr.shape[1]; m = min(sub, len(fr), len(ff))
    g = torch.Generator().manual_seed(seed); vals = []
    for _ in range(n_sub):
        x = fr[torch.randperm(len(fr), generator=g)[:m]]; y = ff[torch.randperm(len(ff), generator=g)[:m]]
        kxx = (x @ x.T / dd + 1) ** 3; kyy = (y @ y.T / dd + 1) ** 3; kxy = (x @ y.T / dd + 1) ** 3
        mmd = ((kxx.sum() - kxx.diag().sum()) / (m * (m - 1)) + (kyy.sum() - kyy.diag().sum()) / (m * (m - 1))
               - 2 * kxy.mean())
        vals.append(float(mmd))
    return float(np.mean(vals)), float(np.std(vals))

def prec_rec(fr, ff, k=3):
    fr, ff = fr.float(), ff.float()
    rr = torch.cdist(fr, fr).kthvalue(k + 1, dim=1).values
    rf = torch.cdist(ff, ff).kthvalue(k + 1, dim=1).values
    dfr = torch.cdist(ff, fr)
    precision = float((dfr <= rr[None, :]).any(1).float().mean())
    recall = float((dfr.T <= rf[None, :]).any(1).float().mean())
    return precision, recall

if op:
    res["KID_mammo_mean"], res["KID_mammo_std"] = kid_feats(FEAT["real"], FEAT["gen"])
    res["precision_mammo"], res["recall_mammo"] = prec_rec(FEAT["real"], FEAT["gen"])
    log(f"mammography-space KID = {res['KID_mammo_mean']:.4f} +- {res['KID_mammo_std']:.4f} | "
        f"precision {res['precision_mammo']:.3f} recall {res['recall_mammo']:.3f}")

# ---------------- Inception KID / FID ----------------
def rgb8(I):
    return (I.clamp(0, 1) * 255).round().to(torch.uint8).repeat(1, 3, 1, 1)
try:
    from torchmetrics.image.kid import KernelInceptionDistance
    kid = KernelInceptionDistance(subset_size=min(100, len(G), len(R))).to(dev)
    for i in range(0, len(R), 32):
        kid.update(rgb8(R[i:i + 32]).to(dev), real=True)
    for i in range(0, len(G), 32):
        kid.update(rgb8(G[i:i + 32]).to(dev), real=False)
    km, ks = kid.compute()
    res["KID_mean"], res["KID_std"] = float(km), float(ks)
    log(f"KID = {float(km):.4f} +- {float(ks):.4f}")
    try:
        fr_i = torch.cat([f.cpu() for f in kid.real_features]); ff_i = torch.cat([f.cpu() for f in kid.fake_features])
        res["precision_inception"], res["recall_inception"] = prec_rec(fr_i, ff_i)
        log(f"Inception precision {res['precision_inception']:.3f} recall {res['recall_inception']:.3f}")
    except Exception as e:
        log("Inception precision/recall skipped:", repr(e)[:150])
except Exception as e:
    log("KID skipped:", repr(e)[:200])
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    fid = FrechetInceptionDistance(feature=2048).to(dev)
    for i in range(0, len(R), 32):
        fid.update(rgb8(R[i:i + 32]).to(dev), real=True)
    for i in range(0, len(G), 32):
        fid.update(rgb8(G[i:i + 32]).to(dev), real=False)
    res["FID"] = float(fid.compute())
    log(f"FID = {res['FID']:.2f}")
except Exception as e:
    log("FID skipped:", repr(e)[:200])

# ---------------- save ----------------
from PIL import Image
k = min(8, a.n_per_class)
rows = [np.concatenate([Gu8[j + c * a.n_per_class] for j in range(k)], axis=1) for c in (0, 1)]
Image.fromarray(np.concatenate(rows, axis=0)).save(os.path.join(a.out, f"grid_{a.arm}_final.png"))
np.savez_compressed(os.path.join(a.out, f"samples_{a.arm}_final.npz"),
                    x=Gu8, y=L.numpy(), correct=correct, phys=rG.numpy())
res["correct"] = [bool(c) for c in correct]
res["labels"] = [int(v) for v in L.numpy()]
with open(os.path.join(a.out, f"results_{a.arm}_final.json"), "w") as fh:
    json.dump(res, fh, indent=1)
log(f"EVAL DONE -> {a.out}/results_{a.arm}_final.json ({(time.time() - t0) / 60:.1f} min)")
''')
Path("classify.py").write_text(r'''# classify.py - DOWNSTREAM UTILITY TEST. Train a benign/malignant classifier (ResNet-18) on
#   (a) the same 25% REAL subset the generators saw, optionally (b) + synthetic images from one generator,
# then test on the untouched REAL CBIS-DDSM test set (AUC, balanced accuracy).
import os, sys, json, time, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

ap = argparse.ArgumentParser()
ap.add_argument("--synth", default="")                   # path to samples_<arm>_epXXX.npz ("" = real only)
ap.add_argument("--tag", required=True)                  # e.g. real_only | base3_s42 | pinn3_s42
ap.add_argument("--clf_seed", type=int, default=0)
ap.add_argument("--data_frac", type=float, default=0.25)
ap.add_argument("--epochs", type=int, default=20)
ap.add_argument("--bs", type=int, default=32)
ap.add_argument("--lr", type=float, default=1e-4)
ap.add_argument("--max_batches", type=int, default=0)    # smoke test only
ap.add_argument("--out", default="/kaggle/working/downstream")
a = ap.parse_args()
dev = torch.device("cuda")
os.makedirs(a.out, exist_ok=True)
TAG = f"[clf-{a.tag}-c{a.clf_seed}]"
def log(*m):
    print(TAG, *m, flush=True)

# same stratified subset as train_arm3.py (rng 123)
d = np.load("/tmp/train_cache.npz")
rng = np.random.default_rng(123); keep = []
for c in (0, 1):
    idx = np.where(d["y"] == c)[0]
    keep += sorted(rng.choice(idx, int(round(a.data_frac * len(idx))), replace=False).tolist())
keep = np.array(sorted(keep))
Xtr, Ytr = d["x"][keep], d["y"][keep]
n_real = len(Ytr)
if a.synth:
    s = np.load(a.synth)
    Xtr = np.concatenate([Xtr, s["x"]]); Ytr = np.concatenate([Ytr, s["y"]])
t = np.load("/tmp/test_cache.npz")
Xte, Yte = t["x"], t["y"]
log(f"train: {n_real} real + {len(Ytr) - n_real} synthetic | test: {len(Yte)} real")

def prep(xu8):                                           # per-image min-max -> [-1,1], 3 channels
    x = torch.from_numpy(xu8).float().unsqueeze(1)
    mn = x.amin(dim=(1, 2, 3), keepdim=True); mx = x.amax(dim=(1, 2, 3), keepdim=True)
    x = ((x - mn) / (mx - mn + 1e-8) - 0.5) / 0.5
    return x.repeat(1, 3, 1, 1)

torch.manual_seed(a.clf_seed); np.random.seed(a.clf_seed)
import torchvision
try:
    net = torchvision.models.resnet18(weights="IMAGENET1K_V1")
    log("ImageNet-pretrained ResNet-18")
except Exception as e:
    net = torchvision.models.resnet18(weights=None)
    log("pretrained weights unavailable -> random init:", repr(e)[:100])
net.fc = nn.Linear(net.fc.in_features, 2)
net = net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=a.lr, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda")
Xt, Yt = prep(Xtr), torch.from_numpy(Ytr).long()
g = torch.Generator().manual_seed(a.clf_seed)
t0 = time.time()
for ep in range(a.epochs):
    net.train()
    perm = torch.randperm(len(Yt), generator=g)
    nb = 0
    for i in range(0, len(perm), a.bs):
        idx = perm[i:i + a.bs]
        x, y = Xt[idx].to(dev), Yt[idx].to(dev)
        flip = torch.rand(len(idx), generator=g) < 0.5
        x = torch.where(flip.to(dev).view(-1, 1, 1, 1), x.flip(-1), x)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = F.cross_entropy(net(x), y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        nb += 1
        if a.max_batches and nb >= a.max_batches:
            break

net.eval(); probs = []
Xe = prep(Xte)
with torch.no_grad():
    for i in range(0, len(Xe), 64):
        with torch.autocast("cuda", dtype=torch.float16):
            probs.append(torch.softmax(net(Xe[i:i + 64].to(dev)).float(), 1)[:, 1].cpu())
p = torch.cat(probs).numpy()
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
auc = float(roc_auc_score(Yte, p)); bacc = float(balanced_accuracy_score(Yte, (p > 0.5).astype(int)))
res = {"tag": a.tag, "clf_seed": a.clf_seed, "synth": a.synth, "n_real": int(n_real),
       "n_synth": int(len(Ytr) - n_real), "epochs": a.epochs, "auc": auc, "balanced_acc": bacc}
with open(os.path.join(a.out, f"clf_{a.tag}_c{a.clf_seed}.json"), "w") as fh:
    json.dump(res, fh, indent=2)
log(f"DONE AUC={auc:.4f} balanced_acc={bacc:.4f} ({(time.time() - t0) / 60:.1f} min)")
''')
Path("summary_gen.py").write_text(r'''# summary_gen.py - Experiment 5 analysis: does the physics-informed loss help when the MONAI U-Net is REPLACED?
#   per generator  : baseline vs physics, paired by seed (descriptive: n = 3 seeds per generator)
#   PRE-REGISTERED : pooled over the 3 new generators x 3 seeds (n = 9 pairs), one-sided Wilcoxon signed-rank on the
#                    RELATIVE paired difference (physics - baseline) / baseline.
#                    PRIMARY   KID in mammography feature space (physics lower)
#                    SECONDARY precision in mammography feature space (physics higher), NPS distance (physics lower)
#                              -> Holm correction over the 2 secondary tests
#   supplementary  : + the MONAI U-Net of Experiment 3 (seeds 42-44) when its evaluation output is attached (n = 12)
# Works with whatever is available (per-model mode on each account, combined mode on the summary account).
import os, sys, json, glob, math, argparse
import numpy as np
from scipy import stats

ap = argparse.ArgumentParser()
ap.add_argument("--models", default="adm,dit,wgan")
ap.add_argument("--seeds", default="42,43,44")
ap.add_argument("--out", default="/kaggle/working/final5")
ap.add_argument("--roots", default="/kaggle/working,/kaggle/input")
a = ap.parse_args()
MODELS = [m for m in a.models.split(",") if m]
SEEDS = [int(s) for s in a.seeds.split(",")]
ROOTS = [r for r in a.roots.split(",") if r]
OUT = a.out
os.makedirs(OUT, exist_ok=True)
NICE = {"monai": "MONAI U-Net (Exp 3, published)", "adm": "ADM U-Net (diffusers)", "dit": "DiT-B/8 transformer",
        "wgan": "conditional WGAN-GP (repo)"}

_index = {}
for root in ROOTS:
    for f in glob.glob(f"{root}/**/*", recursive=True):
        b = os.path.basename(f)
        if b not in _index and os.path.isfile(f):
            _index[b] = f
def find(name):
    return _index.get(name)

def arm_names(model, s):
    if model == "monai":
        return f"base3_s{s}", f"pinn3_s{s}", "ep045"
    return f"{model}_base_s{s}", f"{model}_pinn_s{s}", "final"

def raps(xu8):
    x = xu8.astype(np.float32)
    mn = x.min(axis=(1, 2), keepdims=True); mx = x.max(axis=(1, 2), keepdims=True)
    x = (x - mn) / (mx - mn + 1e-8); x = x - x.mean(axis=(1, 2), keepdims=True)
    P = np.abs(np.fft.fftshift(np.fft.fft2(x), axes=(1, 2))) ** 2
    h, w = x.shape[1:]; yy, xx = np.indices((h, w)); r = np.hypot(yy - h / 2, xx - w / 2).astype(int)
    nb = min(h, w) // 2
    out = np.stack([np.bincount(r.ravel(), weights=p.ravel(), minlength=r.max() + 1)[:nb] /
                    np.bincount(r.ravel(), minlength=r.max() + 1)[:nb] for p in P])
    return np.log10(out.mean(0) + 1e-12)

RAPS_REAL = None
for f in _index:
    if f.startswith("results_") and f.endswith("_final.json"):
        try:
            RAPS_REAL = np.array(json.load(open(_index[f]))["raps_real"]); break
        except Exception:
            pass

def load(model, arm, suffix):
    f = find(f"results_{arm}_{suffix}.json")
    if not f:
        return None
    r = json.load(open(f))
    if "oracle_acc" in r:
        r["oracle_mean"] = r["oracle_acc"]["mean"]; r["oracle_benign"] = r["oracle_acc"]["benign"]
        r["oracle_malignant"] = r["oracle_acc"]["malignant"]
    if ("NPS_dist" not in r or "correct" not in r) and find(f"samples_{arm}_{suffix}.npz"):
        z = np.load(find(f"samples_{arm}_{suffix}.npz"))
        if "NPS_dist" not in r and RAPS_REAL is not None:
            rg = raps(z["x"]); r["NPS_dist"] = float(np.mean((rg[32:128] - RAPS_REAL[32:128]) ** 2))
            r["raps_gen"] = rg.tolist()
        if "correct" not in r:
            r["correct"] = z["correct"].astype(bool).tolist()
    return r

KEYS = [("KID_mammo_mean", "KID mammography", False), ("precision_mammo", "precision (mammo)", True),
        ("recall_mammo", "recall (mammo)", True), ("KID_mean", "KID Inception", False), ("FID", "FID Inception", False),
        ("precision_inception", "precision (Inception)", True), ("recall_inception", "recall (Inception)", True),
        ("physics_abs_gap_to_real", "|NLF residual - real|", False), ("frac_inside_band", "inside real NLF band", True),
        ("NPS_dist", "noise-power-spectrum dist", False), ("oracle_mean", "oracle class accuracy", True),
        ("oracle_benign", "oracle acc benign", True)]

def boot_ci(d, n=20000, seed=0):
    rng = np.random.default_rng(seed); m = rng.choice(d, (n, len(d)), replace=True).mean(1)
    return float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))

def aucs(tag):
    vals = [json.load(open(_index[f]))["auc"] for f in sorted(_index) if f.startswith(f"clf_{tag}_c") and f.endswith(".json")]
    return vals

# ---------------- collect ----------------
TAB, AVAIL = {}, {}
for model in MODELS + ["monai"]:
    ok = []
    for s in SEEDS:
        ab, ap_, suf = arm_names(model, s)
        rb, rp = load(model, ab, suf), load(model, ap_, suf)
        if rb and rp:
            TAB[(model, "base", s)], TAB[(model, "pinn", s)] = rb, rp; ok.append(s)
    if ok:
        AVAIL[model] = ok
print("available (model: seeds with BOTH arms evaluated):", AVAIL)
print("missing:", {m: [s for s in SEEDS if s not in AVAIL.get(m, [])] for m in MODELS})
if not AVAIL:
    sys.exit("nothing evaluated yet")

report = {"available": AVAIL, "per_seed": {f"{m}_{arm}_s{s}": {k: r.get(k) for k, _, _ in KEYS}
                                           for (m, arm, s), r in TAB.items()}}
lines = []
def out(s=""):
    print(s); lines.append(s)

# ---------------- per generator ----------------
for model in [m for m in MODELS + ["monai"] if m in AVAIL]:
    seeds = AVAIL[model]
    rb0 = TAB[(model, "base", seeds[0])]
    out("\n" + "=" * 122)
    npar = rb0.get('n_params')
    out(f"{NICE.get(model, model)} | seeds {seeds} | epochs {rb0.get('epoch')} | params " + (f"{npar / 1e6:.1f}M" if npar else "n/a"))
    out(f"{'metric':28s}{'baseline':>17s}{'physics':>17s}{'diff':>10s}{'better':>8s}{'d_z':>7s}{'Wilcoxon p (2s)':>16s}{'95% boot CI of diff':>22s}")
    block = {}
    for k, name, hb in KEYS:
        b = np.array([TAB[(model, "base", s)].get(k, np.nan) for s in seeds], float)
        p = np.array([TAB[(model, "pinn", s)].get(k, np.nan) for s in seeds], float)
        dd = p - b
        if not np.all(np.isfinite(dd)):
            continue
        better = int(np.sum(dd > 0) if hb else np.sum(dd < 0))
        sd = dd.std(ddof=1) if len(dd) > 1 else np.nan
        dz = dd.mean() / sd if sd and sd > 1e-12 else np.nan
        try:
            pw = float(stats.wilcoxon(p, b).pvalue) if len(dd) >= 2 and np.any(dd != 0) else np.nan
        except Exception:
            pw = np.nan
        lo, hi = boot_ci(dd)
        sb = f"{b.mean():.4f}+-{b.std(ddof=1) if len(b) > 1 else 0:.4f}"
        sp = f"{p.mean():.4f}+-{p.std(ddof=1) if len(p) > 1 else 0:.4f}"
        out(f"{name:28s}{sb:>17s}{sp:>17s}{dd.mean():>+10.4f}{f'{better}/{len(dd)}':>8s}{dz:>7.2f}{pw:>16.4f}"
            f"{f'[{lo:+.4f}, {hi:+.4f}]':>22s}")
        block[k] = {"base": b.tolist(), "physics": p.tolist(), "mean_diff": float(dd.mean()), "d_z": float(dz),
                    "wilcoxon_p_two_sided": pw, "boot_ci": [lo, hi], "n_better": better}
    # McNemar on oracle correctness (paired samples: same noise for both arms)
    if all("correct" in TAB[(model, arm, s)] for s in seeds for arm in ("base", "pinn")):
        cb = np.concatenate([np.array(TAB[(model, "base", s)]["correct"], bool) for s in seeds])
        cp = np.concatenate([np.array(TAB[(model, "pinn", s)]["correct"], bool) for s in seeds])
        b_, c_ = int((cb & ~cp).sum()), int((~cb & cp).sum()); n_ = b_ + c_
        p_mc = min(1.0, 2 * sum(math.comb(n_, j) for j in range(min(b_, c_) + 1)) / 2 ** n_) if n_ else 1.0
        out(f"McNemar oracle correctness ({len(cb)} paired samples): baseline-only correct={b_}, physics-only correct={c_}, p={p_mc:.4f}")
        block["mcnemar"] = {"b": b_, "c": c_, "p": p_mc}
    # downstream
    tb, tp = (("base3", "pinn3") if model == "monai" else (f"{model}_base", f"{model}_pinn"))
    ds = {s: (np.mean(aucs(f"{tb}_s{s}")), np.mean(aucs(f"{tp}_s{s}"))) for s in seeds
          if aucs(f"{tb}_s{s}") and aucs(f"{tp}_s{s}")}
    real = aucs("real_only")
    if ds:
        B = np.array([v[0] for v in ds.values()]); P = np.array([v[1] for v in ds.values()])
        out(f"downstream ResNet-18 AUC on REAL test: real-only {np.mean(real) if real else float('nan'):.4f} | "
            f"+baseline synthetic {B.mean():.4f} | +physics synthetic {P.mean():.4f} | physics better in "
            f"{int(np.sum(P > B))}/{len(P)} seeds")
        block["downstream"] = {"real_only": real, "per_seed": {str(s): list(v) for s, v in ds.items()}}
    report[model] = block

# ---------------- PRE-REGISTERED pooled tests across the NEW generators ----------------
def rel_diffs(models, k, hb):
    rd = []
    for m in models:
        for s in AVAIL.get(m, []):
            b, p = TAB[(m, "base", s)].get(k), TAB[(m, "pinn", s)].get(k)
            if b is None or p is None or not np.isfinite(b) or not np.isfinite(p) or b == 0:
                continue
            rd.append((p - b) / abs(b))
    return np.array(rd)

def pooled_test(models, k, hb):
    rd = rel_diffs(models, k, hb)
    if len(rd) < 2 or not np.any(rd != 0):
        return {"n": int(len(rd)), "p_one_sided": float("nan")}
    w = stats.wilcoxon(rd, alternative=("greater" if hb else "less"))
    return {"n": int(len(rd)), "median_rel_diff": float(np.median(rd)), "mean_rel_diff": float(rd.mean()),
            "n_favour_physics": int(np.sum(rd > 0) if hb else np.sum(rd < 0)), "p_one_sided": float(w.pvalue)}

new = [m for m in MODELS if m in AVAIL]
n_pairs = sum(len(AVAIL[m]) for m in new)
out("\n" + "=" * 122)
out(f"PRE-REGISTERED POOLED TESTS over the new generators {new} ({n_pairs} paired seeds; one-sided Wilcoxon on relative differences)")
complete = (set(new) == {"adm", "dit", "wgan"} and n_pairs == 9)
if not complete:
    out("  (not all 3 generators x 3 seeds available yet -> numbers below are PRELIMINARY, not the confirmatory test)")
pre = {}
for k, name, hb, role in (("KID_mammo_mean", "KID mammography (physics lower)", False, "PRIMARY"),
                          ("precision_mammo", "precision mammography (physics higher)", True, "SECONDARY"),
                          ("NPS_dist", "noise-power-spectrum distance (physics lower)", False, "SECONDARY")):
    r = pooled_test(new, k, hb); r["role"] = role; pre[k] = r
sec = sorted([k for k in pre if pre[k]["role"] == "SECONDARY" and np.isfinite(pre[k]["p_one_sided"])],
             key=lambda k: pre[k]["p_one_sided"])
for i, k in enumerate(sec):                                       # Holm step-down over the secondary tests
    pre[k]["p_holm"] = min(1.0, max(pre[kk]["p_one_sided"] * (len(sec) - j) for j, kk in enumerate(sec[:i + 1])))
for k, name, hb, role in (("KID_mammo_mean", "KID mammography (physics lower)", False, "PRIMARY"),
                          ("precision_mammo", "precision mammography (physics higher)", True, "SECONDARY"),
                          ("NPS_dist", "noise-power-spectrum distance (physics lower)", False, "SECONDARY")):
    r = pre[k]
    if not np.isfinite(r["p_one_sided"]):
        out(f"{role:9s} {name:46s} n={r['n']} -> not enough data"); continue
    pp = r.get("p_holm", r["p_one_sided"])
    verdict = ("MET" if pp < 0.05 else "NOT MET") if complete else "preliminary"
    out(f"{role:9s} {name:46s} n={r['n']}  median rel. diff {100 * r['median_rel_diff']:+.1f}%  "
        f"favour physics {r['n_favour_physics']}/{r['n']}  p={r['p_one_sided']:.4f}"
        + (f" (Holm {pp:.4f})" if "p_holm" in r else "") + f"  -> {verdict}")
report["preregistered"] = {"complete": complete, **pre}
gen_consistency = {m: float(np.mean(rel_diffs([m], "KID_mammo_mean", False))) for m in new
                   if len(rel_diffs([m], "KID_mammo_mean", False))}
out("architecture consistency (mean relative KID-mammo change per generator, negative = physics better): "
    + ", ".join(f"{m} {100 * v:+.1f}%" for m, v in gen_consistency.items()))
report["per_generator_mean_rel_kid_mammo"] = gen_consistency
if "monai" in AVAIL and new:
    allm = new + ["monai"]
    r = pooled_test(allm, "KID_mammo_mean", False)
    out(f"(supplementary) all generators incl. MONAI U-Net of Exp 3, KID mammography: n={r['n']} "
        f"favour physics {r.get('n_favour_physics')}/{r['n']} p={r['p_one_sided']:.4f}")
    report["supplementary_with_monai"] = r
if len(new) >= 2:
    groups = [rel_diffs([m], "KID_mammo_mean", False) for m in new]
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) >= 2:
        kw = stats.kruskal(*groups)
        out(f"(exploratory) does the effect differ between generators? Kruskal-Wallis on relative KID-mammo change: p={kw.pvalue:.4f}")
        report["kruskal_between_generators"] = float(kw.pvalue)

json.dump(report, open(f"{OUT}/summary_exp5.json", "w"), indent=1, default=float)
open(f"{OUT}/summary_exp5.txt", "w").write("\n".join(lines) + "\n")
with open(f"{OUT}/per_seed_table_exp5.csv", "w") as fh:
    fh.write("generator,arm,seed," + ",".join(k for k, _, _ in KEYS) + "\n")
    for (m, arm, s), r in sorted(TAB.items(), key=lambda x: (x[0][0], x[0][2], x[0][1])):
        fh.write(f"{m},{arm},{s}," + ",".join(f"{float(r.get(k, float('nan'))):.5f}" for k, _, _ in KEYS) + "\n")

# ---------------- figures ----------------
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
models = [m for m in ["monai"] + MODELS if m in AVAIL]
show = [("KID_mammo_mean", "KID mammography", False), ("precision_mammo", "precision mammography", True),
        ("NPS_dist", "noise-power-spectrum distance", False), ("physics_abs_gap_to_real", "|NLF residual - real|", False)]
fig, ax = plt.subplots(1, len(show), figsize=(4.6 * len(show), 4.4))
for i, (k, ttl, hb) in enumerate(show):
    for j, m in enumerate(models):
        rd = 100 * rel_diffs([m], k, hb)
        if not len(rd):
            continue
        good = (rd > 0) if hb else (rd < 0)
        ax[i].scatter([j] * len(rd), rd, s=55, c=["tab:green" if g else "tab:red" for g in good], zorder=3)
        ax[i].hlines(rd.mean(), j - 0.28, j + 0.28, color="k", lw=2, zorder=4)
    ax[i].axhline(0, ls="--", c="gray")
    ax[i].set_xticks(range(len(models))); ax[i].set_xticklabels(models)
    ax[i].set_title(f"{ttl}\n({'higher' if hb else 'lower'} = better)", fontsize=10)
ax[0].set_ylabel("relative change, physics vs baseline (%)")
plt.suptitle("Effect of the physics loss per generator (dot = one seed, bar = mean; green = physics better, red = worse)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_relative_change_by_generator.png", dpi=140); plt.close()

if RAPS_REAL is not None:
    fig, ax = plt.subplots(1, len(models), figsize=(4.5 * len(models), 3.8), squeeze=False)
    for j, m in enumerate(models):
        a_ = ax[0, j]; a_.plot(RAPS_REAL, "k", lw=2, label="real test")
        for arm, c in (("base", "tab:blue"), ("pinn", "tab:orange")):
            cur = [TAB[(m, arm, s)].get("raps_gen") for s in AVAIL[m] if TAB[(m, arm, s)].get("raps_gen")]
            if cur:
                a_.plot(np.mean(cur, 0), c, label="physics" if arm == "pinn" else "baseline")
        a_.set_title(m); a_.set_xlabel("radial frequency bin"); a_.legend(fontsize=8)
    ax[0, 0].set_ylabel("log10 power")
    plt.tight_layout(); plt.savefig(f"{OUT}/fig_nps_by_generator.png", dpi=140); plt.close()

try:
    from PIL import Image
    rows = []
    for m in models:
        s = AVAIL[m][0]
        ab, ap_, suf = arm_names(m, s)
        fb, fp = find(f"grid_{ab}_{suf}.png"), find(f"grid_{ap_}_{suf}.png")
        if fb and fp:
            gb, gp = np.array(Image.open(fb).convert("L")), np.array(Image.open(fp).convert("L"))
            rows.append((m, s, gb, gp))
    if rows:
        fig, ax = plt.subplots(len(rows), 2, figsize=(16, 4.2 * len(rows)), squeeze=False)
        for i, (m, s, gb, gp) in enumerate(rows):
            for j, (g, t) in enumerate(((gb, "baseline"), (gp, "physics"))):
                ax[i, j].imshow(g, cmap="gray", vmin=0, vmax=255); ax[i, j].axis("off")
                ax[i, j].set_title(f"{NICE.get(m, m)} - {t} (seed {s}; top row benign, bottom malignant)", fontsize=9)
        plt.tight_layout(); plt.savefig(f"{OUT}/fig_samples_by_generator.png", dpi=110); plt.close()
except Exception as e:
    print("sample figure skipped:", repr(e)[:120])
print(f"\nSaved in {OUT}: summary_exp5.txt/.json, per_seed_table_exp5.csv, fig_relative_change_by_generator.png, "
      f"fig_nps_by_generator.png, fig_samples_by_generator.png")
''')
print("wrote pinn_physics.py, gen_models.py, train_gen.py, eval_gen.py, classify.py, summary_gen.py")

NGPU = 0
if MODEL != "summary":
    # ---------- data cache (identical to Experiments 3-4) ----------
    Path("/tmp/prepare.py").write_text(r'''
import os, subprocess
import numpy as np, pandas as pd
from PIL import Image
if os.path.exists("/tmp/train_cache.npz") and os.path.exists("/tmp/test_cache.npz"):
    print("caches already built"); raise SystemExit(0)
hit = subprocess.run("find /kaggle/input -maxdepth 6 -name dicom_info.csv", shell=True,
                     capture_output=True, text=True).stdout.split()
if not hit:
    raise SystemExit("CBIS-DDSM not attached: Add Input -> CBIS-DDSM Breast Cancer Image Dataset")
RAW = os.path.dirname(os.path.dirname(hit[0]))
info = pd.read_csv(os.path.join(RAW, "csv/dicom_info.csv"))
lookup = info[info["SeriesDescription"] == "full mammogram images"].set_index("PatientName")["image_path"].to_dict()
for split, csvs in [("train", ["csv/calc_case_description_train_set.csv", "csv/mass_case_description_train_set.csv"]),
                    ("test",  ["csv/calc_case_description_test_set.csv",  "csv/mass_case_description_test_set.csv"])]:
    xs, skipped = {0: [], 1: []}, 0
    for c in csvs:
        for _, row in pd.read_csv(os.path.join(RAW, c)).iterrows():
            lab = 1 if row["pathology"] == "MALIGNANT" else 0
            rel = lookup.get(row["image file path"].split("/")[0])
            if rel is None:
                skipped += 1; continue
            p = os.path.join(RAW, "jpeg", rel.replace("CBIS-DDSM/jpeg/", ""))
            if not os.path.exists(p):
                skipped += 1; continue
            xs[lab].append(np.array(Image.open(p).convert("L").resize((256, 256)), dtype=np.uint8))
    if split == "train" and len(xs[0]) > 1300:          # same balancing as before: 1300 benign, all malignant
        keep = np.sort(np.random.default_rng(42).choice(len(xs[0]), 1300, replace=False))
        xs[0] = [xs[0][i] for i in keep]
    x = np.stack(xs[0] + xs[1]); y = np.array([0] * len(xs[0]) + [1] * len(xs[1]), dtype=np.int64)
    np.savez(f"/tmp/{split}_cache.npz", x=x, y=y)
    print(f"{split}: {x.shape} benign={int((y == 0).sum())} malignant={int((y == 1).sum())} skipped={skipped}")
''')
    subprocess.run([sys.executable, "/tmp/prepare.py"], check=True)
    # ---------- inputs and GPU ----------
    ORACLE = glob.glob("/kaggle/input/**/oracle-checkpoint-final/last.pt", recursive=True)
    print("oracle checkpoint:", ORACLE[0] if ORACLE else
          "!! NOT FOUND -> Add Input -> oracle-checkpoint-final (needed for the primary metric; smoke test will FAIL)")
    print(subprocess.run("nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv",
                         shell=True, capture_output=True, text=True).stdout)
    r = subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__); print(torch.cuda.device_count())"],
                       capture_output=True, text=True)
    try:
        NGPU = int(r.stdout.split()[-1])
        print("torch", r.stdout.split()[0], "| GPUs visible:", NGPU)
    except Exception:
        print("!! torch check failed:", r.stderr[-300:])
    if NGPU == 0:
        print("!! NO GPU -> Settings -> Accelerator -> GPU T4 x2, then run again")
    elif NGPU == 1:
        print("!! only 1 GPU -> both arms will run one after the other (twice as slow). Choose GPU T4 x2.")

    # ---------- carry forward: copy this model's earlier results from attached outputs into /kaggle/working ----------
    # (so every new version's output contains ALL results so far, not only this session's)
    pats = {"ckpt": [f"{MODEL}_*_done.json", f"{MODEL}_*_w_final.pt", f"{MODEL}_*_history.json",
                     f"nlf_fit_{MODEL}_*.json", f"{MODEL}_*_last.pt"],
            "eval": [f"results_{MODEL}_*", f"samples_{MODEL}_*", f"grid_{MODEL}_*"],
            "downstream": [f"clf_{MODEL}_*", "clf_real_only_*"],
            "plan": [f"plan_{MODEL}.json"]}
    copied = []
    for sub, pp in pats.items():
        os.makedirs(f"/kaggle/working/{sub}", exist_ok=True)
        for pat in pp:
            for f in glob.glob(f"/kaggle/input/**/{pat}", recursive=True):
                b = os.path.basename(f)
                dst = f"/kaggle/working/{sub}/{b}"
                if "_sm_" in b or "_sm." in b or os.path.exists(dst):
                    continue                                   # smoke-test leftovers / already here
                if b.endswith("_last.pt") and glob.glob(f"/kaggle/input/**/{b[:-8]}_done.json", recursive=True):
                    continue                                   # finished arm: resume file not needed
                shutil.copy(f, dst)
                copied.append(f"{b} ({os.path.getsize(dst) / 1e6:.0f} MB)")
    print(f"carried forward {len(copied)} files from earlier sessions" + (": " + ", ".join(copied[:30]) if copied else ""))

def run_parallel(jobs, poll_min=20):
    """jobs = [(gpu_index, [script, args...]), ...] -> runs them at the same time, one per GPU."""
    if NGPU < 2 and len(jobs) > 1:                     # fallback: one GPU -> one after the other
        rcs, outs = [], []
        for g, argv in jobs:
            rc, o = run_parallel([(0, argv)], poll_min)
            rcs += rc; outs += o
        return rcs, outs
    procs, outs = [], []
    for gpu, argv in jobs:
        env = os.environ.copy()
        env["CUDA_VISIBLE_DEVICES"] = str(gpu if NGPU >= 2 else 0)
        env["PYTHONUNBUFFERED"] = "1"
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        procs.append(p); outs.append([])
    def pump(p, buf, gpu):
        for line in p.stdout:
            buf.append(line)
            print(line if line.startswith("[") else f"[gpu{gpu}] {line}", end="", flush=True)
    def poll():
        while any(p.poll() is None for p in procs):
            time.sleep(poll_min * 60)
            if any(p.poll() is None for p in procs):
                q = subprocess.run("nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv,noheader",
                                   shell=True, capture_output=True, text=True).stdout.strip().replace("\n", " | ")
                print(f"[nvidia-smi] {q} | session time {used_h():.2f} h", flush=True)
    ths = [threading.Thread(target=pump, args=(p, b, g)) for p, b, (g, _) in zip(procs, outs, jobs)]
    for th in ths: th.start()
    threading.Thread(target=poll, daemon=True).start()
    for th in ths: th.join()
    return [p.wait() for p in procs], ["".join(b) for b in outs]
print("Cell 3 done.")

In [ ]:
# ===================== CELL 4 — SMOKE TEST + TIME PLAN (first session only, ~15 min) =====================
# Tests: model builds, training, exact resume, atomic saving, physics term active only in the physics arm,
# evaluation with all metrics, downstream classifier. Measures the speed and writes plan_<MODEL>.json
# (number of epochs, checkpoint interval, time estimates) that ALL later sessions reuse.
import os, re, glob, shutil, json, time
SMOKE_OK, PLAN = False, None
if MODEL != "summary":
    C = MCFG[MODEL]
    pf = find_any(f"plan_{MODEL}.json")
    if pf:
        PLAN = json.load(open(pf)); SMOKE_OK = True
        print("smoke test passed in an earlier session -> skipped | plan:", PLAN)
    elif NGPU == 0:
        print("no GPU -> smoke test not run")
    else:
        SM, EV, DS = "/kaggle/working/ckpt_smoke", "/kaggle/working/eval_smoke", "/kaggle/working/downstream_smoke"
        for p in (SM, EV, DS): shutil.rmtree(p, ignore_errors=True)
        lam = LAM_GAN if MODEL == "wgan" else LAM_DIFF
        extra = (["--n_critic", str(C["n_critic"]), "--gp", str(C["gp"]), "--gan_iters", str(C["gan_iters"])]
                 if MODEL == "wgan" else ["--accum", str(C["accum"])])
        common = ["--model", MODEL, "--data_frac", str(DATA_FRAC), "--lr", str(C["lr"]), "--bs", str(C["bs"]),
                  "--smoke_batches", "30", "--ckpt_dir", SM, "--ckpt_every", "1", "--log_every", "10"] + extra
        def train_jobs(ep):
            return [(0, ["train_gen.py", "--arm", f"{MODEL}_base_sm", "--lam", "0.0", "--epochs", str(ep)] + common),
                    (1, ["train_gen.py", "--arm", f"{MODEL}_pinn_sm", "--lam", str(lam), "--epochs", str(ep)] + common)]
        ev_args = ["--n_per_class", "8", "--steps", "5", "--bs", "8", "--ckpt_dir", SM, "--out", EV, "--require_oracle"]
        t_s = time.time()
        rc1, o1 = run_parallel(train_jobs(1), poll_min=5)
        rc2, o2 = run_parallel(train_jobs(2), poll_min=5) if rc1 == [0, 0] else ([1, 1], ["", ""])
        rc3, o3 = (run_parallel([(0, ["eval_gen.py", "--arm", f"{MODEL}_base_sm"] + ev_args),
                                 (1, ["eval_gen.py", "--arm", f"{MODEL}_pinn_sm"] + ev_args)], poll_min=5)
                   if rc2 == [0, 0] else ([1, 1], ["", ""]))
        clf_args = ["--data_frac", str(DATA_FRAC), "--epochs", "1", "--max_batches", "3", "--out", DS]
        rc4, o4 = (run_parallel([(0, ["classify.py", "--tag", "real_only"] + clf_args),
                                 (1, ["classify.py", "--tag", f"{MODEL}_pinn_sm", "--synth",
                                      f"{EV}/samples_{MODEL}_pinn_sm_final.npz"] + clf_args)], poll_min=5)
                   if rc3 == [0, 0] else ([1, 1], ["", ""]))
        S = r"START epoch=(\d+) step=(\d+) opt_states=(\d+) lam=\S+ checksum=(\S+)"
        def g(pat, s):
            m = re.search(pat, s); return m.groups() if m else None
        def rj(p):
            return json.load(open(p))
        def hist(arm):
            return rj(f"{SM}/{arm}_history.json")[-1]
        B, P_ = f"{MODEL}_base_sm", f"{MODEL}_pinn_sm"
        checks = []
        for k, arm in enumerate([B, P_]):
            checks += [
                (f"{arm}: run 1 started from scratch",          lambda k=k: g(S, o1[k])[:2] == ("0", "0")),
                (f"{arm}: run 2 RESUMED at epoch 1, step 30",  lambda k=k: g(S, o2[k])[:2] == ("1", "30")),
                (f"{arm}: run 2 optimizer state restored",     lambda k=k: int(g(S, o2[k])[2]) > 0),
                (f"{arm}: files written atomically",
                 lambda arm=arm: all(os.path.exists(f"{SM}/{arm}_{x}") for x in ["last.pt", "w_final.pt", "history.json"])
                                 and not glob.glob(f"{SM}/*.tmp")),
                (f"{arm}: speed measured",                     lambda arm=arm: rj(f"{SM}/timing_{arm}.json")["sec_per_it"] > 0),
                (f"{arm}: evaluation with mammography metrics + NPS",
                 lambda arm=arm: all(x in rj(f"{EV}/results_{arm}_final.json") for x in ("KID_mammo_mean", "NPS_dist", "KID_mean"))),
            ]
        checks += [
            ("both arms start from IDENTICAL weights",             lambda: g(S, o1[0])[3] == g(S, o1[1])[3]),
            ("baseline loss == main loss (physics NOT trained)",   lambda: abs(hist(B)["loss"] - hist(B)["main"]) <= 1e-6 * max(1.0, abs(hist(B)["main"]))),
            ("physics arm loss > main loss (physics term active)", lambda: hist(P_)["loss"] - hist(P_)["main"] > 0),
            ("downstream classifier: real only",                    lambda: "auc" in rj(f"{DS}/clf_real_only_c0.json")),
            ("downstream classifier: real + synthetic",             lambda: rj(f"{DS}/clf_{MODEL}_pinn_sm_c0.json")["n_synth"] == 16),
        ]
        def ok(f):
            try: return bool(f())
            except Exception: return False
        print("\n" + "=" * 70)
        for name, f in checks:
            print(("PASS  " if ok(f) else "FAIL  ") + name)
        SMOKE_OK = all(ok(f) for _, f in checks)
        if SMOKE_OK:
            tb, tp = rj(f"{SM}/timing_{B}.json"), rj(f"{SM}/timing_{P_}.json")
            sec = max(tb["sec_per_it"], tp["sec_per_it"]); spe = tb["spe_full"]
            ep_min = sec * spe / 60
            note = "epoch-matched to the MONAI U-Net (45 epochs)" if C["kind"] == "diffusion" else \
                   f"compute-matched to the MONAI U-Net ({C['pair_h']} GPU-hours per seed pair)"
            if C["kind"] == "diffusion":
                epochs = C["epochs"]
                if epochs * ep_min * 1.06 / 60 > MAX_PAIR_H:
                    epochs = max(10, int(C["epochs"] * MAX_PAIR_H / (epochs * ep_min * 1.06 / 60)))
                    note = f"!! slower than expected -> reduced to {epochs} epochs to fit {MAX_PAIR_H} h per pair"
                eval_h = 25 * 50 * 4.6 * sec / 3600 + 0.12
            else:
                lo = min(30, int(MAX_PAIR_H * 60 / (ep_min * 1.04)))      # at least 3000 generator steps if they fit
                epochs = int(max(lo, min(1000, C["pair_h"] * 60 / (ep_min * 1.04))))
                note += f" = {epochs * spe} generator steps"
                eval_h = 0.15
            pair_h = epochs * ep_min * 1.06 / 60
            PLAN = {"model": MODEL, "epochs": epochs, "sec_per_it": sec, "iters_per_epoch": spe,
                    "epoch_min": ep_min, "ckpt_every": max(1, round(15 / max(ep_min, 1e-3))),
                    "pair_h_est": pair_h, "eval_h_est": eval_h, "peak_mem_gb": max(tb["peak_mem_gb"], tp["peak_mem_gb"]),
                    "note": note, "created": time.strftime("%Y-%m-%d %H:%M UTC", time.gmtime())}
            os.makedirs("/kaggle/working/plan", exist_ok=True)
            json.dump(PLAN, open(f"/kaggle/working/plan/plan_{MODEL}.json", "w"), indent=2)
            for p in (SM, EV, DS): shutil.rmtree(p, ignore_errors=True)
            total = 3 * (pair_h + eval_h) + 0.6
            print(f"ALL PASS ({(time.time() - t_s) / 60:.0f} min) -> smoke files deleted, Cell 5 will train.")
            print(f"PLAN: {epochs} epochs x {spe} iterations ({ep_min:.1f} min/epoch) | ~{pair_h:.1f} h training + "
                  f"~{eval_h:.1f} h evaluation per seed | ~{total:.1f} GPU-hours in total for 3 seeds | {note}")
        else:
            print("SMOKE TEST FAILED -> Cell 5 will not train. Send Claude this block + the first error lines above.")

In [ ]:
# ===================== CELL 5 — MAIN PIPELINE: for each seed  train -> evaluate -> downstream (resumable) =====================
import os, json
def is_done(arm):  return bool(find_any(f"{arm}_done.json"))
def is_eval(arm):  return bool(find_any(f"results_{arm}_final.json"))
def has_clf(tag, c): return bool(find_any(f"clf_{tag}_c{c}.json"))
def run_clf(jobs):
    for i in range(0, len(jobs), 2):
        if used_h() + 0.15 > BUDGET_H:
            print("out of session time -> remaining classifier runs next session"); return False
        batch = jobs[i:i + 2]
        rc, _ = run_parallel([(g, ["classify.py", "--tag", tag, "--clf_seed", str(c), "--data_frac", str(DATA_FRAC)]
                               + (["--synth", npz] if npz else [])) for g, (tag, npz, c) in enumerate(batch)], poll_min=15)
        if any(rc):
            print("!! classifier run failed, exit codes", rc); return False
    return True

if MODEL == "summary":
    print("summary mode -> nothing to train")
elif not SMOKE_OK:
    print("Smoke test did not pass -> nothing will run. Send Claude the smoke-test block of Cell 4.")
else:
    C, P = MCFG[MODEL], PLAN
    extra = (["--n_critic", str(C["n_critic"]), "--gp", str(C["gp"]), "--gan_iters", str(C["gan_iters"])]
             if MODEL == "wgan" else ["--accum", str(C["accum"])])
    for s in SEEDS:
        arms = ARMS(s)
        # ---------- 1. train both arms in parallel (GPU 0 = baseline, GPU 1 = physics) ----------
        todo = [x for x in arms if not is_done(x[1])]
        if todo:
            if used_h() + 0.7 > BUDGET_H:
                print(f"seed {s}: not enough session time left to train -> next session"); break
            jobs = [(gpu, ["train_gen.py", "--model", MODEL, "--arm", arm, "--lam", str(lam), "--seed", str(s),
                           "--epochs", str(P["epochs"]), "--data_frac", str(DATA_FRAC), "--lr", str(C["lr"]),
                           "--bs", str(C["bs"]), "--budget_h", str(BUDGET_H), "--ckpt_every", str(P["ckpt_every"]),
                           "--log_every", str(C["log_every"]), "--final"] + extra) for gpu, arm, lam in todo]
            print(f"\n########## seed {s}: training {[a for _, a, _ in todo]} ({P['epochs']} epochs, ~{P['pair_h_est']:.1f} h) ##########")
            rc, _ = run_parallel(jobs, poll_min=30)
            print(f"seed {s} training exit codes: {rc}")
            if not all(is_done(a) for _, a, _ in arms):
                print(f"seed {s}: training not finished in this session (time budget or error above) -> continues next session")
                break
        # ---------- 2. evaluate (same noise for both arms -> paired) ----------
        todo = [a for _, a, _ in arms if not is_eval(a)]
        if todo:
            if used_h() + P["eval_h_est"] + 0.1 > BUDGET_H:
                print(f"seed {s}: not enough session time to evaluate (~{P['eval_h_est']:.1f} h) -> next session"); break
            print(f"\n########## seed {s}: evaluating {todo} ##########")
            rc, _ = run_parallel([(g, ["eval_gen.py", "--arm", a, "--require_oracle"]) for g, a in enumerate(todo)], poll_min=20)
            if not all(is_eval(a) for _, a, _ in arms):
                print(f"!! seed {s}: evaluation failed (exit codes {rc}) - see the error above"); break
        # ---------- 3. downstream classifier (real 25% + 400 synthetic, AUC on the real test set) ----------
        jobs = [("real_only", "", c) for c in CLF_SEEDS if not has_clf("real_only", c)]
        jobs += [(a, find_any(f"samples_{a}_final.npz"), c) for _, a, _ in arms for c in CLF_SEEDS if not has_clf(a, c)]
        if jobs:
            print(f"\n########## seed {s}: {len(jobs)} downstream classifier runs ##########")
            if not run_clf(jobs):
                break
        print(f"========== seed {s} COMPLETE (session time {used_h():.2f} h of {BUDGET_H} h) ==========")
    st = {f"s{s}": ("done" if all(is_eval(a) and all(has_clf(a, c) for c in CLF_SEEDS) for _, a, _ in ARMS(s))
                    else "trained" if all(is_done(a) for _, a, _ in ARMS(s)) else "in progress / not started")
          for s in SEEDS}
    print("STATUS:", st)

In [ ]:
# ===================== CELL 6 — results of THIS model + share zip for the summary account =====================
import os, sys, glob, zipfile, subprocess
if MODEL != "summary":
    if any(find_any(f"results_{a}_final.json") for s in SEEDS for _, a, _ in ARMS(s)):
        OUT6 = f"/kaggle/working/final5_{MODEL}"
        r = subprocess.run([sys.executable, "summary_gen.py", "--models", MODEL, "--out", OUT6], cwd=WD,
                           capture_output=True, text=True)
        print(r.stdout[-15000:]); print(r.stderr[-3000:])
        from IPython.display import Image, display
        for f in ("fig_samples_by_generator.png", "fig_relative_change_by_generator.png", "fig_nps_by_generator.png"):
            if os.path.exists(f"{OUT6}/{f}"):
                display(Image(f"{OUT6}/{f}"))
    pats = [f"results_{MODEL}_*_final.json", f"grid_{MODEL}_*_final.png", f"clf_{MODEL}_*.json", "clf_real_only_c*.json",
            f"{MODEL}_*_history.json", f"{MODEL}_*_done.json", f"nlf_fit_{MODEL}_*.json", f"plan_{MODEL}.json"]
    files = sorted({f for p in pats for f in glob.glob(f"/kaggle/working/**/{p}", recursive=True)
                    if "_sm_" not in os.path.basename(f) and "_sm." not in os.path.basename(f)})
    files += sorted(glob.glob(f"/kaggle/working/final5_{MODEL}/*"))
    zp = f"/kaggle/working/share_{MODEL}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        seen = set()
        for f in files:
            b = os.path.basename(f)
            if b not in seen:
                seen.add(b); z.write(f, f"share_{MODEL}/{b}")
    complete = all(find_any(f"results_{a}_final.json") and all(find_any(f"clf_{a}_c{c}.json") for c in CLF_SEEDS)
                   for s in SEEDS for _, a, _ in ARMS(s))
    print(f"\n{zp}: {len(seen)} files, {os.path.getsize(zp) / 1e6:.1f} MB")
    print("SHARE FILE COMPLETE -> download it from the Output tab and send it to the summary account" if complete else
          "share file is PARTIAL (not all seeds finished) -> next session: attach THIS version's output and Save & Run All again")
print(f"session time used: {used_h():.2f} h of {BUDGET_H} h")

In [ ]:
# ===================== CELL 7 — COMBINED ANALYSIS of all generators (MODEL = "summary" only) =====================
# Inputs: a Kaggle dataset containing share_adm.zip, share_dit.zip, share_wgan.zip (Kaggle unpacks zips on upload).
# Optional: the Experiment-3 EVALUATION output (adds the MONAI U-Net as a 4th generator, supplementary).
import os, sys, glob, zipfile, subprocess
if MODEL == "summary":
    os.makedirs("/kaggle/working/combined", exist_ok=True)
    for zf in glob.glob("/kaggle/input/**/share_*.zip", recursive=True):     # zips that were NOT unpacked
        with zipfile.ZipFile(zf) as z:
            z.extractall("/kaggle/working/combined")
        print("extracted", zf)
    for m in ("adm", "dit", "wgan"):
        n = len(glob.glob(f"/kaggle/input/**/results_{m}_*_final.json", recursive=True) +
                glob.glob(f"/kaggle/working/combined/**/results_{m}_*_final.json", recursive=True))
        print(f"{m}: {n} evaluated arms found (6 expected)")
    OUT7 = "/kaggle/working/final5_ALL"
    r = subprocess.run([sys.executable, "summary_gen.py", "--models", "adm,dit,wgan", "--out", OUT7,
                        "--roots", "/kaggle/working/combined,/kaggle/input"], cwd=WD, capture_output=True, text=True)
    print(r.stdout[-25000:]); print(r.stderr[-3000:])
    from IPython.display import Image, display
    for f in ("fig_samples_by_generator.png", "fig_relative_change_by_generator.png", "fig_nps_by_generator.png"):
        if os.path.exists(f"{OUT7}/{f}"):
            display(Image(f"{OUT7}/{f}"))
    print(f"DONE. Download /kaggle/working/final5_ALL/* (summary_exp5.txt is the table for the paper)")
else:
    print('(combined analysis runs only with MODEL = "summary")')